# Reproducing Li, Daie, Svoboda and Druckmann (2016) from dandiset 000011

Paper. Li N, Daie K, Svoboda K, Druckmann S (2016). Robust neuronal dynamics in premotor cortex
during motor planning. *Nature* 532, 459-464.
[doi:10.1038/nature17643](https://doi.org/10.1038/nature17643). The claims file
(`claims/000011/10.1038-nature17643.yaml`) was written from the HHMI author manuscript in PubMed
Central (PMC5081260): main text, Methods and the legends of Figures 1 to 6 and Extended Data
Figures 1 to 10. The Corrigendum (doi:10.1038/nature18623) was not read.

Dandiset. [DANDI:000011](https://dandiarchive.org/dandiset/000011/0.220126.1907), version
0.220126.1907. The paper cites no DANDI version (its data statement points to CRCNS,
doi:10.6080/K0RB72JW), so the most recent published version is pinned. It is the only published
version. The draft lists the same 92 assets with the same total size (32,435,325,542 bytes); its
metadata was modified later (2026-02-12), which the analysis does not use.

What the dandiset holds (shown in the data check below). 91 extracellular files of 19 mice and
one empty behaviour file. 58 sessions of 12 mice are the control-mouse recordings of the paper
(experiment 1, unilateral photoinhibition, 20 sessions of 4 mice; experiments 2 and 3, bilateral
photoinhibition, 38 sessions of 7 and 3 mice). 33 sessions of 7 other mice have no photostimulus
record; their subjects and delay match the callosotomy experiment. The behavioural photoinhibition
experiments without recordings, the photoactivation mice and the untrained mice used to map
photoinhibition are not in the dandiset.

| claim | tested or blocked |
|---|---|
| unit-classification | tested |
| selective-fraction | tested |
| fig1d-late-delay-ipsilateral-bias | blocked: data_missing |
| fig1d-earlier-photoinhibition-minimal | blocked: data_missing |
| ed3-behaviour-recovery-interval | blocked: data_missing |
| ed4-return-to-trajectory | tested |
| fig1f-spike-rate-recovery | tested |
| fig1g-selectivity-recovery | tested |
| ed5-photoactivation-recovery | blocked: data_missing |
| ed5-photoactivation-rate-changes | blocked: data_missing |
| ed2b-silencing-strength | tested |
| ed2d-spatial-extent | blocked: data_missing |
| ed2e-no-rebound | tested |
| fig2b-unilateral-performance | blocked: data_missing |
| fig2b-bilateral-performance | blocked: data_missing |
| ed6a-bilateral-inconsistent-bias | tested |
| fig2-recordings-recovery | tested |
| fig2e-selectivity-recovery-times | tested |
| ed7-larger-photoinhibition | tested |
| ed7e-roc-larger-photoinhibition | tested |
| cd-stable-over-epochs | tested |
| cd-captures-selectivity | tested |
| fig3c-cd-recovery-roc | tested |
| ed8-roc-during-photoinhibition | tested |
| fig3c-cd-predicts-movement | tested |
| fig3d-persistent-mode | tested |
| fig3e-ramping-mode | tested |
| fig4-bilateral-cd-predicts-movement | tested |
| ed6b-session-bias-predicted | tested |
| ed9f-reaction-time | blocked: methods_underspecified (question asked) |
| ed9c-cd-window-robust | tested |
| fig5b-callosotomy-control-performance | blocked: data_missing |
| fig5b-callosotomy-unilateral-sensitivity | blocked: metadata_insufficient |
| ed10-callosotomy-preparatory-activity | blocked: metadata_insufficient |
| fig5-callosotomy-no-recovery | blocked: metadata_insufficient |
| fig5f-callosotomy-cd-reduced | blocked: metadata_insufficient |
| ed1-monolithic-models-fail | tested (simulation) |
| fig6-modular-models-recover | tested (simulation) |

Readings. Where the claims file lists readings of a detail, each is run, and a claim is
reproduced when it holds under one of them, as its criterion says. Where the claims file does not
give a detail and the paper does not either, a question was asked (`questions/000011.yaml`) and
every reading of it is run; when those readings give different outcomes, the claim is blocked as
`methods_underspecified` until the question is answered. The open details are: whether lick-early
trials enter the neural analyses (they are kept or dropped; in them the delay restarts and the
photostimulus event is logged at a different time from the light), which spike counts set a
neuron's preferred trial type, whether "more than 3 trials in all conditions" counts each trial type,
which time a window is labelled by when a recovery time is read, and others named in their sections.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import itertools
import time
import warnings

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import remfile
from scipy import stats

import li2016 as L

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
# LOAD: stream every file of the pinned version (trials, units, photostimulus record)
started = time.time()
assets, results = L.load_all()
S, U = L.assemble(results)
print(f"loaded {len(results)} extracellular files of dandiset {L.DANDISET} version {L.VERSION} in {time.time() - started:.0f} s")

## Data check

In [ ]:
print("Assets in the version:", len(assets))
print(assets.assign(kind=np.where(assets.path.str.contains("ecephys"), "behavior+ecephys+ogen", "behavior only"))
      .groupby("kind").size().to_string())
# The one behaviour-only file: list everything it holds besides the format specification
beh = assets[~assets.path.str.contains("ecephys")].iloc[0]
with h5py.File(remfile.File(L.s3_url(beh.asset_id)), "r") as f:
    items = []
    f.visititems(lambda n, o: items.append((n, getattr(o, "shape", None))) if not n.startswith("specifications") else None)
print(f"\n{beh.path}: {len(items)} objects outside /specifications")
for n, sh in items:
    print("  ", n, sh if sh is not None else "(group)")

In [ ]:
ses = pd.DataFrame([{"sid": s["sid"], "subject": s["subject"], "genotype": "VGAT" if "VGAT" in s["genotype"] else "PV x ReaChR",
                     "experiment": s["exp"], "trials": len(s["trials"]),
                     "behaving_trials": int(s["trials"].behaving.sum()),
                     "units": int((U.sid == s["sid"]).sum()), "sites": ",".join(s["meta"]["sites"]),
                     "trials_overlapping": s["meta"]["trials_overlapping"]} for s in S.values()])
by_subject = ses.groupby(["subject", "genotype"]).agg(sessions=("sid", "size"), experiments=("experiment", lambda x: ",".join(sorted(set(x)))),
                                                     units=("units", "sum")).reset_index()
print(by_subject.to_string(index=False))
grp = ses.assign(group=np.where(ses.experiment == "no_photostim_record", "no photostimulus record", "control mice"))
print("\nSessions, mice and units by group:")
print(grp.groupby("group").agg(sessions=("sid", "size"), mice=("subject", "nunique"), units=("units", "sum")).to_string())
print("\nSessions by experiment:")
print(ses.groupby("experiment").agg(sessions=("sid", "size"), mice=("subject", "nunique"), units=("units", "sum")).to_string())
print("\nGenotypes of the control mice:", grp[grp.group == "control mice"].groupby("genotype").subject.nunique().to_dict())
print("Genotypes of the mice without a photostimulus record:", grp[grp.group != "control mice"].groupby("genotype").subject.nunique().to_dict())
print("\nPaper (Methods): 58 sessions (20 unilateral, 38 bilateral) of 12 mice (5 VGAT, 7 PV x ReaChR);"
      " callosotomy 7 mice (6 VGAT, 1 PV x ReaChR); Extended Data Fig. 2b: experiment 1 from 6 mice,"
      " experiment 2 from 7, experiment 3 from 3.")

The subjects split in two groups. 12 mice (5 VGAT-ChR2-EYFP, 7 PV x ReaChR) with 58 sessions are
the control mice of the paper: their counts match the Methods. The 33 sessions of the 7 other
mice (6 VGAT, 1 PV x ReaChR) have no photostimulus record at all; the genotype counts and the
1.7 s delay match the callosotomy experiment, but no file says these mice were callosotomised
(the subject description holds only the JAX strains). Experiment 1 holds 4 mice here, where
Extended Data Fig. 2b gives 6. Three experiment-1 sessions of mouse 255200 have no photostimulus
record either. The behaviour-only file holds only metadata.

In [ ]:
rows = []
for s in S.values():
    tr = s["trials"]
    b = tr[tr.behaving]
    ne = b[~b.early_lick]
    rows.append({"experiment": s["exp"], "sid": s["sid"], "behaving": len(b), "early_lick": int(b.early_lick.sum()),
                 "sample_s_median": np.median(ne.delay - ne["sample"]) if len(ne) else np.nan,
                 "delay_s_median": np.median(ne.go - ne.delay) if len(ne) else np.nan,
                 "delay_off_nominal_noearly": int((abs((ne.go - ne.delay) - np.median(ne.go - ne.delay)) > 0.05).sum()) if len(ne) else 0,
                 "delay_off_nominal_early": int((abs((b[b.early_lick].go - b[b.early_lick].delay) - np.median(ne.go - ne.delay)) > 0.05).sum()) if len(ne) else 0,
                 **{o: int((b.outcome == o).sum()) for o in ["hit", "miss", "ignore"]},
                 "non_performing": int((tr.outcome == "non-performing").sum()),
                 "trials_overlapping": s["meta"]["trials_overlapping"]})
tt = pd.DataFrame(rows)
print("Behaving trials (sample, delay and go events and an outcome); session 258250_20141202 is set to none (below).")
print(tt.groupby("experiment").agg(sessions=("sid", "size"), behaving=("behaving", "sum"), early_lick=("early_lick", "sum"),
                                   hit=("hit", "sum"), miss=("miss", "sum"), ignore=("ignore", "sum"), non_performing=("non_performing", "sum"),
                                   sample_s=("sample_s_median", lambda x: f"{x.min():.2f}-{x.max():.2f}"),
                                   delay_s=("delay_s_median", lambda x: f"{x.min():.2f}-{x.max():.2f}"),
                                   off_nominal_noearly=("delay_off_nominal_noearly", "sum"),
                                   off_nominal_early=("delay_off_nominal_early", "sum")).to_string())
print("\nSessions whose trials overlap in time:")
print(tt[tt.trials_overlapping > 0][["sid", "trials_overlapping"]].to_string(index=False))
bad = S["258250_20141202"]["trials"]
print("\n258250_20141202, trial ids in time order (first 30 behaving or not):", bad.trial_id.values[:30].tolist())
print("task and protocol values over all files:", sorted(set(np.concatenate([s["trials"].task.unique() for s in S.values()]))),
      sorted(set(np.concatenate([s["trials"].protocol.unique() for s in S.values()]))))

In every session the sample epoch (pole onset to delay onset) lasts 1.3 s and the delay 1.3 s
(experiment 1) or 1.7 s (experiments 2, 3 and the sessions without a photostimulus record), as the
Methods say, in trials without an early lick. In early-lick trials the delay restarts, so their
delay differs from the nominal one. Session 258250_20141202 holds two interleaved sequences of
trial ids whose times overlap (98 overlapping pairs), so its trial events cannot be assigned to
trials; it is left out of every trial-based analysis and its units are counted only. Two sessions of
mouse 255200 have one or two overlapping pairs and are kept. The trial
table has no column for the photostimulus: conditions are read from the laser record below.

In [ ]:
ps = []
for s in S.values():
    m = s["meta"]
    if not m["sites"]:
        continue
    tr = s["trials"]
    ps.append({"sid": s["sid"], "experiment": s["exp"], "events": m["ps_events"], "events_without_time": m["ps_events_nan_time"],
               "laser_chunks": sum(m[f"chunks_{x}"] for x in m["sites"]),
               "chunks_outside_trials": sum(m[f"chunks_outside_{x}"] for x in m["sites"]),
               "chunk_trials_without_event": int((tr.ps_site.notna() & tr.ps_on.isna()).sum()),
               "of_them_behaving": int((tr.ps_site.notna() & tr.ps_on.isna() & tr.behaving).sum()),
               "event_trials_without_chunk": int((tr.ps_site.isna() & tr.ps_on.notna()).sum()),
               "samples_per_chunk": {x: m[f"samples_{x}"] / m[f"chunks_{x}"] for x in m["sites"]}})
ps = pd.DataFrame(ps)
print(ps.drop(columns="samples_per_chunk").groupby("experiment").sum(numeric_only=True).to_string())
print("Samples per laser chunk (all sites, all sessions):", sorted(set(v for d in ps.samples_per_chunk for v in d.values())))
print("Sessions with event trials lacking a laser chunk:", ps[ps.event_trials_without_chunk > 0][["sid", "event_trials_without_chunk"]].values.tolist())
shape = []
for s in S.values():
    for k, v in s["meta"].items():
        if k.startswith("shape_"):
            shape.append({"experiment": s["exp"], "site": k[6:], "mean_mW": round(v[0], 2), "cv": round(v[1], 2)})
shape = pd.DataFrame(shape)
print("\nLaser power 0.1-0.5 s after onset (mW) in the first trial of each site and session:")
print(shape.groupby(["experiment", "site"]).agg(sessions=("cv", "size"), mean_mW=("mean_mW", lambda x: f"{x.min()}-{x.max()}"),
                                                cv=("cv", lambda x: f"{x.min()}-{x.max()}")).to_string())
allt = pd.concat([s["trials"].assign(experiment=s["exp"], sid=s["sid"]) for s in S.values() if s["meta"]["sites"]], ignore_index=True)
light = allt[allt.ps_on.notna()]
print("\nPeak power of photostimulus events (mW), counts:", np.histogram(light.ps_power, bins=[0, 1, 2.5, 3.5, 5, 7, 11, 14])[0].tolist(),
      "for bins [0, 1, 2.5, 3.5, 5, 7, 11, 14]")

The photostimulus of a trial is recorded twice: as `photostim_start_time` and
`photostim_stop_time` events (the event value is the peak power, mW) and as 5 s chunks of the laser
power and AOM traces (10 kHz, 50,000 samples per chunk), one chunk per photostimulus trial, in one
series per site (left, right, bilateral). The chunks give the site; the events give the time. The
two agree trial by trial: every event with a time has a chunk except in two sessions, and the
chunks without an event fall in non-performing trials. Event values below 1 mW (77 events) are
kept apart as `no_light`. The traces separate the protocols of the Methods: a 40 Hz sinusoid of
1.5 mW average for one spot, and a constant power of 1.5 mW per spot when the beam is scanned
(3.1 mW for one spot per side, 6.1 mW for four spots on one side, 12.6 mW for four spots per
side). Experiment 2 sessions use one spot left or right and eight spots bilaterally; experiment 3
sessions use two spots bilaterally (one per side) and four spots left or right, or one spot left
or right in 6 of its 12 sessions.

In [ ]:
lab = allt[allt.behaving & ~allt.early_lick]
print("Behaving trials without an early lick, by experiment and condition:")
print(pd.crosstab(lab.cond, lab.experiment).to_string())
lp = lab[lab.ps_on.notna() & ~lab.cond.isin(["no_light", "ps_unknown"])]
print("\nLight onset and offset re go cue (s), and duration, by condition (trials without an early lick):")
print(lp.groupby(["experiment", "cond"]).agg(n=("ps_on", "size"), onset=("ps_on_re_go", lambda x: f"{x.min():.3f} to {x.max():.3f}"),
                                             offset=("ps_off_re_go", lambda x: f"{x.min():.3f} to {x.max():.3f}"),
                                             duration=("ps_off", lambda x: "")).drop(columns="duration").to_string())
print("Light duration (s), all conditions:", np.unique(np.round(lp.ps_off - lp.ps_on, 3)).tolist())
print("Onset re delay onset (s) in experiments 2 and 3:", np.unique(np.round((lp.ps_on - lp.delay)[lp.experiment != 'exp1'], 3)).tolist())
print("Fraction of behaving trials with light, per session (control sessions):",
      allt[allt.behaving].groupby("sid").apply(lambda d: (~d.cond.isin(["control"])).mean()).describe().round(2).to_dict())

Experiment 1 holds three conditions, 0.5 s of light ending 1.6 s (sample), 0.8 s (early delay) and
0.3 s (late delay) before the go cue. This settles the naming question of the claims file
(convention T): the recorded "late delay" condition ends 0.3 s before the go cue (the Methods'
naming); the condition ending at the go cue (the Results' "late delay" of Fig. 1d) is not in the
recordings. Experiments 2 and 3 have 0.8 s of light from delay onset. The stop event is 0.5 s or
0.8 s after the start event, so it marks the end of the ramp, the offset the claims file uses.

In [ ]:
# Event time against the light in the trace: two sessions read in full
for sid in ["255200_20140913", "263808_20150121"]:
    s = S[sid]; tr = s["trials"]
    aid = assets[assets.path == s["path"]].asset_id.iloc[0]
    rows = []
    with h5py.File(remfile.File(L.s3_url(aid)), "r") as f:
        for site in s["meta"]["sites"]:
            g = f[f"/stimulus/presentation/{site} ALM_laser_power"]
            d, ts = g["data"][:], g["timestamps"][:]
            for r in tr[(tr.ps_site == site) & tr.ps_on.notna() & tr.behaving].itertuples():
                c = int(r.ps_chunk); seg = d[c * 50000:(c + 1) * 50000]; t = ts[c * 50000:(c + 1) * 50000]
                on = np.where(seg > 0.5)[0]
                rows.append({"early_lick": r.early_lick, "event_minus_light_ms": (r.ps_on - t[on[0]]) * 1000 if len(on) else np.nan,
                             "delay_event_minus_light_ms": (r.delay - t[on[0]]) * 1000 if len(on) else np.nan})
    df = pd.DataFrame(rows)
    df["mismatch"] = df.event_minus_light_ms.abs() > 20
    print(sid, "trials:", len(df))
    print(df.groupby("early_lick").agg(n=("mismatch", "size"), mismatch=("mismatch", "sum"),
                                       event_minus_light_ms=("event_minus_light_ms", lambda x: f"{x.min():.0f} to {x.max():.0f}")).to_string())

In trials without an early lick the photostimulus event and the light in the trace agree to
within 5 ms. In early-lick trials the event is logged up to about 2 s after the light, at the
restarted delay, so in those trials the event times do not mark the light. This is one reason
the readings below that keep early-lick trials are kept apart.

In [ ]:
ctl_units = U[U.exp != "no_photostim_record"]
print("Units in control mice:", len(ctl_units), "; without a photostimulus record:", int((U.exp == "no_photostim_record").sum()))
print("\nWidth class (trough to peak of waveform_mean, Methods thresholds) against the files' cell_type:")
print(pd.crosstab(U.cls, U.cell_type).to_string())
print("\nTrough-to-peak widths (ms) and counts, control mice:")
w = ctl_units.width_ms.round(3).value_counts().sort_index()
print(w.to_string())
print("\nWaveform samples:", sorted(set(len(x) for x in U.waveform)), "; sampling rate (Hz):", sorted(set(U.fs)),
      "; Methods: 19531.25 Hz")
print("Waveform_mean range (stated unit not given in the file):", f"{np.min([x.min() for x in U.waveform]):.2e} to {np.max([x.max() for x in U.waveform]):.2e}")
print("Unit quality values:", U.quality.value_counts().to_dict())
print("Units whose spike times are not in time order in the file:", {s["sid"]: s["meta"]["units_unsorted"] for s in S.values() if s["meta"]["units_unsorted"]},
      "; sessions whose trials are not in time order:", {s["sid"]: s["meta"]["trials_unsorted"] for s in S.values() if s["meta"]["trials_unsorted"]})
outside = []
for u in U.itertuples():
    obs = u.obs
    inside = np.zeros(len(u.spikes), bool)
    for a, b in obs:
        inside |= (u.spikes >= a) & (u.spikes <= b)
    outside.append((~inside).mean())
print("Fraction of spikes outside the unit's observation intervals: max", f"{max(outside):.4f}")
obs_frac = [o[S[s]["trials"].behaving.values].mean() for o, s in zip(U.observed, U.sid) if S[s]["trials"].behaving.any()]
print("Fraction of behaving trials in which a unit is observed: median", f"{np.median(obs_frac):.2f}", "min", f"{np.min(obs_frac):.2f}")
print("Mean firing rate over observation intervals (Hz), control mice:",
      pd.Series([len(sp) / max(np.sum(o[:, 1] - o[:, 0]), 1e-9) for sp, o in zip(ctl_units.spikes, ctl_units.obs)]).describe().round(2).to_dict())

The files' `cell_type` labels follow the same thresholds as the Methods (every unit wider than
0.45 ms is `Pyr`), except that the units of intermediate width are labelled `FS` (45 in all; the 15
in the control mice are all 0.358 ms wide). At 19531.25 Hz the width takes discrete values 0.051 ms apart; none falls on a threshold.
In 28 units of 3 sessions the spike times are not in time order (the files store them in trial
order, and two of these sessions have trials out of time order); they are sorted here. A few spikes
(at most 3.3% of a unit's) fall outside the unit's observation intervals.

In [ ]:
# Clock check 1: spikes against the photostimulus record. Population rate in the light window
# (onset to offset minus ramp) over the control rate in the same window, and the same with the
# window shifted 1.5 s later (after the light).
rows = []
for sid, s in S.items():
    if s["exp"] not in ("exp1", "exp2", "exp3") or s["meta"]["trials_overlapping"]:
        continue
    tr = s["trials"]; m = L.trial_mask(tr, "drop")
    uu = U[(U.sid == sid) & (U.cls == "pyr")]
    ipsi = {"exp1": ["sample", "early_delay", "late_delay"], "exp2": ["left1"], "exp3": ["left4", "bilateral2"]}[s["exp"]]
    for c in ipsi:
        p = m & (tr.cond == c).values; k = m & (tr.cond == "control").values
        if p.sum() < 3:
            continue
        on, off = np.median(tr.ps_on_re_go[p]), np.median(tr.ps_off_re_go[p]) - (0.1 if s["exp"] == "exp1" else 0.2)
        for shift in (0.0, 1.5):
            a = sum(L.counts_in(u.spikes, tr.go.values[p], on + shift, off + shift).mean() for u in uu.itertuples())
            b = sum(L.counts_in(u.spikes, tr.go.values[k], on + shift, off + shift).mean() for u in uu.itertuples())
            rows.append({"sid": sid, "experiment": s["exp"], "cond": c, "shift_s": shift, "ratio": a / b if b else np.nan})
clk = pd.DataFrame(rows)
print("Population rate, photoinhibition over control, in the light window and 1.5 s later (sessions):")
print(clk.groupby(["experiment", "cond", "shift_s"]).ratio.describe()[["count", "min", "50%", "max"]].round(2).to_string())

In [ ]:
# Clock check 2: trial type against spikes. Fraction of putative pyramidal units whose delay-epoch
# counts differ between lick left and lick right control trials (t-test, p < 0.05), with the true
# trial types and with the trial types of the next trial (a shift that breaks the link).
rows = []
for sid, s in S.items():
    if s["meta"]["trials_overlapping"] or s["exp"] == "no_photostim_record" and False:
        continue
    tr = s["trials"]; m = L.trial_mask(tr, "drop", ("hit",)) & tr.cond.isin(["control", "unlabelled"]).values
    if m.sum() < 20:
        continue
    t = tr[m]; right = (t.instruction == "right").values
    shifted = np.roll(right, 1)
    uu = U[(U.sid == sid) & (U.cls == "pyr")]
    for u in uu.itertuples():
        ok = u.observed[m]
        c = L.counts_in(u.spikes, t.go.values, (t.delay - t.go).values, np.zeros(len(t)))
        rows.append({"sid": sid, "experiment": s["exp"], "true": L.ttest(c[ok & right], c[ok & ~right]) < 0.05,
                     "shifted": L.ttest(c[ok & shifted], c[ok & ~shifted]) < 0.05})
dec = pd.DataFrame(rows)
print("Fraction of units selective in the delay epoch, true against shifted trial types:")
print(dec.groupby("experiment")[["true", "shifted"]].mean().round(3).assign(units=dec.groupby("experiment").size()).to_string())

In [ ]:
# Sessions without a photostimulus record: per-trial population rate in the window where the light
# falls in the other sessions (experiment 1: the three windows; otherwise delay onset + 0.0-0.6 s),
# over the session median. Trials below 0.3 are counted; the same count 1.5 s later is the reference.
rows = []
for sid, s in S.items():
    if s["exp"] not in ("no_photostim_record", "exp1_unlabelled"):
        continue
    tr = s["trials"]; m = L.trial_mask(tr, "drop", ("hit", "miss", "ignore"))
    t = tr[m]
    uu = U[(U.sid == sid)]
    if len(t) < 10 or len(uu) == 0:
        rows.append({"sid": sid, "experiment": s["exp"], "trials": len(t)}); continue
    if s["exp"] == "exp1_unlabelled":
        wins = [(-2.1, -1.7), (-1.3, -0.9), (-0.8, -0.4)]
    else:
        wins = [(-1.7, -1.1)]
    out = {"sid": sid, "experiment": s["exp"], "trials": len(t), "units": len(uu)}
    for shift, name in ((0.0, "silenced"), (1.5, "silenced_1.5s_later")):
        low = np.zeros(len(t), bool)
        for a, b in wins:
            pop = sum(L.counts_in(u.spikes, t.go.values, a + shift, b + shift) for u in uu.itertuples())
            med = np.median(pop)
            low |= pop < 0.3 * med if med > 0 else np.zeros(len(t), bool)
        out[name] = round(low.mean(), 2)
    rows.append(out)
unl = pd.DataFrame(rows)
print(unl.to_string(index=False))
print("\nMedian over sessions of the fraction of trials below 30% of the session median, in the light window and 1.5 s later:")
print(unl.groupby("experiment")[["silenced", "silenced_1.5s_later"]].median().to_string())
print("Sessions where the fraction in the light window exceeds the one 1.5 s later by 0.1 or more:",
      int(((unl.silenced - unl["silenced_1.5s_later"]) >= 0.1).sum()), "of", int(unl.silenced.notna().sum()))

The clock checks hold for the 54 sessions with a photostimulus record. In each ipsilateral
condition the population rate in the light window has a median over sessions of 0.03 to 0.05 of the
control rate (one experiment-2 session reaches 0.70, and one experiment-3 session 1.26 under
bilateral light), and 1.5 s later the median is 0.94 to 1.09, so the spikes, the trial events and the
photostimulus record share one clock. Delay-epoch selectivity is found in 0.30 to 0.39 of units with
the true trial types against 0.06 to 0.09 with those of the next trial, in every group except the
three unlabelled experiment-1 sessions (0.33 against 0.23, 30 units).

In most sessions without a photostimulus record, a fraction of trials (up to about 0.4) have their
population rate fall below 30% of the session median in the light window of the other sessions,
far more than 1.5 s later. These sessions therefore contain photoinhibition trials that no file
labels (in a few sessions no such trials appear in this window, so their light, if any, fell
elsewhere). Their
control trials cannot be told from their photoinhibition trials, so no analysis of them that needs
control trials or conditions can be run.

## unit-classification

> 1012 single units were recorded in left ALM during behaviour in 58 sessions (20 unilateral, 38
> bilateral experiment sessions); by spike width 72 were fast-spiking, 890 putative pyramidal, and
> 50 intermediate and excluded.

Criterion: width is the trough-to-peak interval of the mean waveform; units and sessions of the
control mice only. Reproduced when all values are within tolerance.

In [ ]:
ctl = U[U.exp.isin(["exp1", "exp1_unlabelled", "exp2", "exp3"])]
expected = {"single units": (1012, 101.2), "recording sessions": (58, 5.8), "fast-spiking units": (72, 7.2),
            "putative pyramidal units": (890, 89), "intermediate units": (50, 5)}
measured = {"single units": len(ctl), "recording sessions": ctl.sid.nunique(), "fast-spiking units": int((ctl.cls == "FS").sum()),
            "putative pyramidal units": int((ctl.cls == "pyr").sum()), "intermediate units": int((ctl.cls == "intermediate").sum())}
uc = pd.DataFrame([{"quantity": k, "paper": v[0], "tolerance": v[1], "measured": measured[k],
                    "within": abs(measured[k] - v[0]) <= v[1]} for k, v in expected.items()])
print(uc.to_string(index=False))
print("\nSessions: unilateral (experiment 1)", ctl[ctl.exp.str.startswith("exp1")].sid.nunique(),
      "; bilateral (experiments 2 and 3)", ctl[ctl.exp.isin(["exp2", "exp3"])].sid.nunique())
print("Units by mouse and class:")
print(pd.crosstab(ctl.subject, ctl.cls, margins=True).to_string())
cal = U[U.exp == "no_photostim_record"]
print("\nCounted apart, mice without a photostimulus record:", len(cal), "units,", cal.sid.nunique(), "sessions;",
      cal.cls.value_counts().to_dict())
print("Units missing against the paper's 1012:", 1012 - len(ctl), "; intermediate units missing against its 50:", 50 - measured["intermediate units"])

**Result: partially reproduced.** The control mice hold 979 units in 58 sessions (20 unilateral, 38
bilateral), with 74 fast-spiking and 890 putative pyramidal units, all within tolerance; 890 is the
paper's number exactly. Only 15 units have intermediate widths, against 50: the dandiset holds 33
units fewer than the paper's 1012, about the number of intermediate units missing (35), so the
intermediate units were probably left out of the deposit.

## selective-fraction

> Most putative pyramidal neurons distinguished trial types (634 of 890); 176 in the sample epoch,
> 337 in the delay epoch and 493 in the response epoch.

Criterion: convention S on the putative pyramidal neurons, control trials; reproduced when all four
counts are within tolerance under one combination of readings: response epoch 1.3 s (R1) or 1.5 s
(R2) after the go cue; correct trials (O1) or correct and incorrect (O2); Student's or Welch's
t-test. Lick-early trials are dropped or kept (question 1). The 44 putative pyramidal units of the
three experiment-1 sessions without a photostimulus record have no identifiable control trials;
they are left out, and shown as a variant with all their trials.

In [ ]:
pyr_ctl = ctl[ctl.cls == "pyr"]
pyr_lab = pyr_ctl[pyr_ctl.exp.isin(["exp1", "exp2", "exp3"])]
SEL, sel_rows, tests = {}, [], {}
for early in ("drop", "keep"):
    SEL[early] = {}
    for r in L.S_READINGS:
        st = L.selectivity_tests(pyr_lab, S, r["response"], r["outcomes"], early, r["equal_var"])
        tests[(early, L.s_key(r))] = st
        sig = st[["sample", "delay", "response"]] < 0.05
        SEL[early][L.s_key(r)] = set(st.uid[sig.any(axis=1)])
        sel_rows.append({"early_lick": early, "reading": L.s_key(r), "tested": len(st),
                         "untestable": int(st[["sample", "delay", "response"]].isna().any(axis=1).sum()),
                         "any": int(sig.any(axis=1).sum()), "sample": int(sig["sample"].sum()),
                         "delay": int(sig.delay.sum()), "response": int(sig.response.sum())})
sf = pd.DataFrame(sel_rows)
exp_sf = {"any": (634, 63.4), "sample": (176, 17.6), "delay": (337, 33.7), "response": (493, 49.3)}
for k, (v, tol) in exp_sf.items():
    sf[k + "_ok"] = (sf[k] - v).abs() <= tol
sf["all_four"] = sf[[k + "_ok" for k in exp_sf]].all(axis=1)
print("Putative pyramidal units with labelled trials:", len(pyr_lab), "of", len(pyr_ctl))
print(sf.to_string(index=False))
print("\nCombinations with all four within tolerance:", int(sf.all_four.sum()), "of", len(sf))
print("Largest sample-epoch count:", sf["sample"].max(), "; tolerance band", 176 - 17.6, "to", 176 + 17.6)

In [ ]:
# Variant: the 44 units of the three unlabelled sessions added, with all their trials as control
rows = []
for early in ("drop", "keep"):
    for r in L.S_READINGS:
        st = L.selectivity_tests(pyr_ctl, S, r["response"], r["outcomes"], early, r["equal_var"], conds=("control", "unlabelled"))
        sig = st[["sample", "delay", "response"]] < 0.05
        row = {"early_lick": early, "reading": L.s_key(r), "any": int(sig.any(axis=1).sum()), "sample": int(sig["sample"].sum()),
               "delay": int(sig.delay.sum()), "response": int(sig.response.sum())}
        row["all_four"] = all(abs(row[k] - v) <= tol for k, (v, tol) in exp_sf.items())
        rows.append(row)
sfv = pd.DataFrame(rows)
print("Variant with the 44 units of the unlabelled sessions (all their trials):")
print(sfv.to_string(index=False))
print("By mouse, reading R1-O1-student, lick-early dropped (labelled sessions):")
st = tests[("drop", "R1-O1-student")].merge(pyr_lab[["uid", "subject"]], on="uid")
print(st.assign(sel=(st[["sample", "delay", "response"]] < 0.05).any(axis=1)).groupby("subject").sel.agg(["sum", "size"]).T.to_string())

**Result: partially reproduced.** On the 846 putative pyramidal units whose control trials can be
identified, no combination of readings brings all four counts within tolerance: the sample-epoch
count is at most 156, below the band of 158 to 194, while the counts in any epoch (up to 588), the
delay epoch (up to 320) and the response epoch (up to 464) reach their bands under some readings.
With the 44 units of the three unlabelled sessions added on all their trials, all four are within
tolerance with correct trials only and lick-early trials dropped (592, 165, 307 and 464 with R1
and Student's test), and under no reading that keeps lick-early trials.

## Behaviour during the recordings (variant for blocked claims)

fig1d-late-delay-ipsilateral-bias, fig1d-earlier-photoinhibition-minimal,
ed3-behaviour-recovery-interval and the two fig2b claims rest on the paper's behavioural
photoinhibition experiments (5 and 13 mice, photoinhibition on 25% of trials, conditions not in the
recordings such as light ending at the go cue, 1.3 s photoinhibition and four-spot unilateral
photoinhibition). Those sessions are not in the dandiset, so the claims are blocked. The behaviour
recorded during the electrophysiology sessions is shown here for reference only; it is a different
set of sessions and does not decide an outcome.

In [ ]:
def perf(tr, cond, side=None, ignore_as_error=False):
    m = tr.behaving & ~tr.early_lick & (tr.cond == cond)
    if not ignore_as_error:
        m &= tr.outcome.isin(["hit", "miss"])
    if side:
        m &= tr.instruction == side
    t = tr[m]
    return (t.outcome == "hit").mean() if len(t) else np.nan, len(t)

rows = []
for s in S.values():
    if s["exp"] not in ("exp1", "exp2"):
        continue
    for c in ["control", "sample", "early_delay", "late_delay", "left1", "right1", "bilateral8"]:
        p, n = perf(s["trials"], c)
        pr, _ = perf(s["trials"], c, "right")
        if n:
            rows.append({"experiment": s["exp"], "subject": s["subject"], "cond": c, "perf": p, "perf_right": pr, "n": n})
bp = pd.DataFrame(rows)
mouse = bp.groupby(["experiment", "subject", "cond"]).apply(lambda d: pd.Series({"perf": np.average(d.perf, weights=d.n), "perf_right": np.nanmean(d.perf_right)})).reset_index()
for exp, conds in [("exp1", ["sample", "early_delay", "late_delay"]), ("exp2", ["left1", "right1", "bilateral8"])]:
    m = mouse[mouse.experiment == exp].pivot(index="subject", columns="cond", values="perf")
    print(f"{exp}: performance per mouse (fraction correct, lick-early excluded)")
    print(m.round(3).to_string())
    for c in conds:
        tp = stats.ttest_rel(m[c], m["control"]).pvalue
        ti = stats.ttest_ind(m[c], m["control"]).pvalue
        print(f"  {c}: mean {m[c].mean():.3f} against control {m['control'].mean():.3f}; paired p = {tp:.3g}, unpaired p = {ti:.3g}")

## ed4-return-to-trajectory

> Neurons whose spike rates decreased during the delay epoch also recovered after unilateral
> photoinhibition, returning to the time-matched control trajectory and not decaying to the
> end-point rate.

Criterion: experiment 1; neurons with a significant decrease of the spike count in the last 400 ms
of the delay against the first 400 ms (two-tailed t-test, p < 0.05, trial types pooled); rates
normalised to the neuron's mean rate; for each condition the RMS between photoinhibition offset and
the go cue of the perturbed minus time-matched control rate and of the perturbed rate minus the
control rate at the end of the delay; readings (a) one value per neuron averaged over conditions
and trial types, (b) one per neuron, condition and trial type; reproduced when the time-matched RMS
is smaller by a paired t-test with p < 0.001 under a reading. Not given and asked (question 6):
whether the selection test is paired (same trials) or unpaired, and the bin of the rate time
course (200 ms, as the PSTHs of Fig. 1e, or 400 ms); each is run, with lick-early trials dropped or
kept. Neurons are the putative pyramidal neurons (convention U); trials are correct and incorrect
ones; the mean rate is over control trials from sample onset to 1.3 s after the go cue.

In [ ]:
OFF1 = {"sample": -1.6, "early_delay": -0.8, "late_delay": -0.3}
e1 = U[(U.exp == "exp1") & (U.cls == "pyr")]
ed4_rows, ed4_detail = [], {}
for early in ("drop", "keep"):
    for paired in (True, False):
        chosen = []
        for u in e1.itertuples():
            tr = S[u.sid]["trials"]
            m = L.trial_mask(tr, early) & (tr.cond == "control").values & u.observed
            t = tr[m]
            a = L.counts_in(u.spikes, t.go.values, (t.delay - t.go).values, (t.delay - t.go).values + 0.4)
            b = L.counts_in(u.spikes, t.go.values, -0.4, 0.0)
            if len(t) < 3:
                continue
            p = stats.ttest_rel(b, a).pvalue if paired else stats.ttest_ind(b, a).pvalue
            if p < 0.05 and b.mean() < a.mean():
                chosen.append(u.uid)
        for w in (0.2, 0.4):
            sub = e1[e1.uid.isin(chosen)]
            rm = L.rate_matrices(sub, S, w, early)
            tl = L.labels(w, "centre")
            j_end = int(np.argmin(abs(L.GRID_END - 0.0)))     # window ending at the go cue
            per = []
            for uid, d in rm.items():
                ctrl = d["cond"] == "control"
                so = d["so"][ctrl]
                n_all = L.counts_in(d["spikes"], d["go"][ctrl], so, np.full(ctrl.sum(), 1.3))
                mean_rate = n_all.sum() / np.sum(1.3 - so)
                if mean_rate <= 0:
                    continue
                for c, off in OFF1.items():
                    win = (tl > off) & (tl <= 0)
                    for side in (True, False):
                        pm = (d["cond"] == c) & (d["right"] == side)
                        cm = ctrl & (d["right"] == side)
                        if pm.sum() < 1 or cm.sum() < 1:
                            continue
                        rp = d["rate"][pm].mean(0) / mean_rate
                        rc = d["rate"][cm].mean(0) / mean_rate
                        per.append({"uid": uid, "cond": c, "right": side,
                                    "rms_matched": np.sqrt(np.mean((rp[win] - rc[win]) ** 2)),
                                    "rms_endpoint": np.sqrt(np.mean((rp[win] - rc[j_end]) ** 2))})
            per = pd.DataFrame(per)
            ed4_detail[(early, paired, w)] = per
            avg = per.groupby("uid")[["rms_matched", "rms_endpoint"]].mean()
            ta = stats.ttest_rel(avg.rms_matched, avg.rms_endpoint)
            tb = stats.ttest_rel(per.rms_matched, per.rms_endpoint)
            ed4_rows.append({"early_lick": early, "selection_test": "paired" if paired else "unpaired", "bin_s": w,
                             "neurons": len(chosen), "mice": sub.subject.nunique(),
                             "a_matched": avg.rms_matched.mean(), "a_endpoint": avg.rms_endpoint.mean(), "a_t": ta.statistic, "a_p": ta.pvalue,
                             "b_n": len(per), "b_matched": per.rms_matched.mean(), "b_endpoint": per.rms_endpoint.mean(), "b_t": tb.statistic, "b_p": tb.pvalue})
ed4 = pd.DataFrame(ed4_rows)
ed4["holds_a"] = (ed4.a_p < 0.001) & (ed4.a_matched < ed4.a_endpoint)
ed4["holds_b"] = (ed4.b_p < 0.001) & (ed4.b_matched < ed4.b_endpoint)
print(ed4.round(4).to_string(index=False))
print("\nPaper: 27 neurons from 6 mice (tolerance 2.7), paired t-test p < 0.001.")

**Result: not reproduced.** The selection finds 47 to 67 putative pyramidal neurons with a
significant delay-epoch decrease, from 4 mice (paper 27 from 6). Under no reading is the RMS of the
difference from the time-matched control smaller than the RMS of the difference from the end-point
rate: the two are within 0.02 of each other in every row (paired p 0.41 to 0.98 with one value per
neuron, 0.30 to 0.98 with one value per neuron, condition and trial type).

## fig1f-spike-rate-recovery

> Within 400 ms after unilateral photoinhibition, spike rates became indistinguishable from control
> in 90% of neurons; 10% retained a sustained change.

Criterion: convention R on putative pyramidal neurons of experiment 1, per condition; the fraction
of neurons with a significant change (p < 0.01) in the 400 ms window at 400 ms after
photoinhibition offset; readings (a) each of the three conditions, (b) the three pooled; window
labelled by its end (W1) or centre (W2); neurons below 1 spike/s or with fewer than 3 trials
excluded (X1) or not (X2). Reproduced when the fraction is within 0.09 to 0.11 under one
combination. Lick-early trials dropped or kept (question 1).

In [ ]:
f1_rows = []
for early in ("drop", "keep"):
    for X in ("X1", "X2"):
        res_c = {}
        for c, off in OFF1.items():
            P, info = L.rate_change_tests(e1, S, c, 0.4, early, min_rate=1 if X == "X1" else None, min_trials=3 if X == "X1" else None)
            res_c[c] = (P, info)
        for lab in ("end", "centre"):
            tl = L.labels(0.4, lab)
            tot_sig = tot_n = 0
            row = {"early_lick": early, "exclusion": X, "label": "W1 end" if lab == "end" else "W2 centre"}
            for c, off in OFF1.items():
                P, info = res_c[c]
                inc = info.included.values & ~np.isnan(P).all(1)
                j = int(np.argmin(abs(tl - (off + 0.4))))
                sig = P[inc, j] < 0.01
                row[f"n_{c}"] = int(inc.sum()); row[f"frac_{c}"] = sig.mean()
                jd = int(np.argmin(abs(tl - off)))
                row[f"during_{c}"] = (P[inc, jd] < 0.01).mean()
                tot_sig += sig.sum(); tot_n += inc.sum()
            row["frac_pooled"] = tot_sig / tot_n
            f1_rows.append(row)
f1 = pd.DataFrame(f1_rows)
f1["a_each_within"] = f1[[f"frac_{c}" for c in OFF1]].sub(0.1).abs().le(0.01 + 1e-12).all(axis=1)
f1["b_pooled_within"] = (f1.frac_pooled - 0.1).abs() <= 0.01 + 1e-12
print(f1.round(3).to_string(index=False))
print("\nPaper: n = 168, 168, 175 neurons; fraction 0.10 (tolerance 0.01). 'during_' is the fraction at the window labelled at offset.")

**Result: reproduced.** With windows labelled by their centre and no exclusion (W2, X2), the pooled
fraction of neurons with a significant change 400 ms after offset is 0.101 with lick-early trials
dropped and 0.097 with them kept, both within 0.09 to 0.11. The per-condition fractions under that
reading are 0.105, 0.113 and 0.085 (dropped), so reading (a) does not hold; with the exclusion (X1)
170 to 172 neurons are tested per condition (paper 168 to 175) and the pooled fraction is 0.112 to
0.165. During the light 27% to 72% of neurons are significant, depending on the window label.

## fig1g-selectivity-recovery

> After unilateral photoinhibition ALM neurons recovered more than 80% of their control selectivity
> within about 0.5 s; recovery times were 373 ms (sample), 510 ms (early delay) and 327 ms (late
> delay).

Criterion: convention D on experiment 1 (selective neurons tested on more than 3 trials in every
condition; preferred trial type from 10 random control trials, redrawn 20 times and averaged;
selectivity on correct and incorrect trials grouped by instruction); recovery time is the first
window after photoinhibition offset where the population selectivity on photoinhibition trials
reaches 80% of the time-matched control selectivity; bins of 200 ms (B1) or 400 ms (B2) in 50 ms
steps. Reproduced when all three times are within the paper's s.e.m. under one bin reading;
partially for two of three. The selective neurons are those of selective-fraction under each of
its readings. Not given and asked (questions 1 to 4): lick-early trials dropped or kept; the
preferred trial type from counts over the whole trial or over the delay epoch; more than 3 trials
per condition or per condition and trial type; windows labelled by their centre or their end. The
search runs to the end of the windows (1.6 s after the go cue), as the late-delay recovery of the
paper (327 ms after an offset 0.3 s before the go cue) falls after the go cue. The measured time is
that of the population of all included neurons; the bootstrap (200 resamples of neurons) mean and
s.d. are shown beside it.

In [ ]:
t0 = time.time()
rec1 = pd.concat([L.recovery_readings(e1, S, OFF1, SEL[early], early)[0] for early in ("drop", "keep")], ignore_index=True)
exp1g = {"sample": (373, 260), "early_delay": (510, 218), "late_delay": (327, 112)}
for c, (v, tol) in exp1g.items():
    rec1[c + "_ok"] = (rec1[c] - v).abs() <= tol
rec1["n_ok"] = (rec1.n - 55).abs() <= 5.5
rec1["times_ok"] = rec1[[c + "_ok" for c in exp1g]].sum(axis=1)
print(f"{len(rec1)} reading combinations in {time.time() - t0:.0f} s")
show = ["early_lick", "pref", "trials_rule", "label", "bin", "S", "n", "sample", "early_delay", "late_delay", "sample_boot", "sample_sd",
        "early_delay_boot", "early_delay_sd", "late_delay_boot", "late_delay_sd", "times_ok"]
print(rec1.rename(columns={"early": "early_lick"})[show].round(0).to_string(index=False))

In [ ]:
g = rec1.rename(columns={"early": "early_lick"}).groupby(["early_lick", "pref", "trials_rule", "label"])
summary1g = g.apply(lambda d: pd.Series({"combos": len(d), "all_three": int((d.times_ok == 3).sum()), "two": int((d.times_ok == 2).sum()),
                                        "n_range": f"{d.n.min()}-{d.n.max()}"})).reset_index()
summary1g["outcome_here"] = np.where(summary1g.all_three > 0, "reproduced", np.where(summary1g.two > 0, "partially", "not"))
print("Per combination of the open details (any S reading and bin reading may match):")
print(summary1g.to_string(index=False))
print("\nIncluded neurons, all readings:", rec1.n.min(), "to", rec1.n.max(), "; paper 55 (tolerance 5.5)")

**Result: blocked (methods_underspecified).** The outcome depends on the details asked about. With
lick-early trials dropped, every combination of the preference window and the window label has
readings with all three times within tolerance (for example, delay-epoch preference and centre
labels: sample 200 to 350 ms, early delay 250 to 350 ms, late delay 300 ms). With lick-early trials
kept, the early-delay time is 650 to 1400 ms under most readings: two of three are within
tolerance with the delay-epoch preference, all three with the whole-trial preference and centre
labels, and none with whole-trial preference and end labels. Every reading includes 138 to 173
neurons, against the paper's 55.

## ed2b-silencing-strength

> Ipsilateral and bilateral photoinhibition reduced the activity of putative pyramidal neurons by
> more than 90%, similarly in VGAT-ChR2-EYFP and PV x ReaChR mice; contralateral photoinhibition
> had little effect on the average rate.

Criterion: per neuron (mean rate at least 1 spike/s), the rate during photoinhibition over the
control rate in the same window; window (a) the whole photostimulus before the ramp, (b) the last
400 ms of photoinhibition. Reproduced when the mean normalised rate is below 0.1 for ipsilateral
photoinhibition in both mouse lines (experiment 2) and for bilateral photoinhibition, and
contralateral photoinhibition leaves it above 0.8. Judged on experiment 2; experiments 1 and 3 are
shown. Neurons need 2 trials in the condition and in control (a standard choice); trials are
correct and incorrect ones; lick-early dropped or kept (question 1).

In [ ]:
CONDS = {"exp1": {"sample": "ipsilateral", "early_delay": "ipsilateral", "late_delay": "ipsilateral"},
         "exp2": {"left1": "ipsilateral", "right1": "contralateral", "bilateral8": "bilateral"},
         "exp3": {"left4": "ipsilateral", "right4": "contralateral", "bilateral2": "bilateral"}}
RAMP = {"exp1": 0.1, "exp2": 0.2, "exp3": 0.2}
rows = []
for early in ("drop", "keep"):
    for exp, cmap in CONDS.items():
        uu = U[(U.exp == exp) & (U.cls == "pyr")]
        for u in uu.itertuples():
            tr = S[u.sid]["trials"]
            m = L.trial_mask(tr, early) & u.observed
            k = m & (tr.cond == "control").values
            so = (tr["sample"] - tr.go).values[k]
            n_all = L.counts_in(u.spikes, tr.go.values[k], so, np.full(k.sum(), 1.3))
            mean_rate = n_all.sum() / np.sum(1.3 - so) if k.sum() else 0
            if mean_rate < 1:
                continue
            for c, kind in cmap.items():
                p = m & (tr.cond == c).values
                if p.sum() < 2 or k.sum() < 2:
                    continue
                on, off = np.median(tr.ps_on_re_go.values[p & ~tr.early_lick.values]), np.median(tr.ps_off_re_go.values[p & ~tr.early_lick.values])
                for wname, (a, b) in {"a": (on, off - RAMP[exp]), "b": (off - 0.4, off)}.items():
                    cp = L.counts_in(u.spikes, tr.go.values[p], a, b).mean()
                    cc = L.counts_in(u.spikes, tr.go.values[k], a, b).mean()
                    rows.append({"early_lick": early, "experiment": exp, "cond": c, "kind": kind, "window": wname,
                                 "genotype": u.genotype, "uid": u.uid, "norm": cp / cc if cc > 0 else np.nan})
nr = pd.DataFrame(rows)
tab = nr.groupby(["early_lick", "window", "experiment", "cond", "kind"]).norm.agg(["size", "mean", "sem"]).round(3)
print(tab.to_string())
print("\nExperiment 2, ipsilateral, by mouse line:")
print(nr[(nr.experiment == "exp2") & (nr.kind == "ipsilateral")].groupby(["early_lick", "window", "genotype"]).norm.agg(["size", "mean", "sem"]).round(3).to_string())
print("\nLegend counts: experiment 1 n = 117, 110, 109; experiment 2 n = 300, 294, 301; experiment 3 n = 52, 52, 102;"
      " experiment 2 ipsilateral 94 VGAT and 201 PV neurons.")
judge = []
for (early, wname), d in nr[nr.experiment == "exp2"].groupby(["early_lick", "window"]):
    ip = d[d.kind == "ipsilateral"].groupby("genotype").norm.mean()
    judge.append({"early_lick": early, "window": wname, "ipsi_VGAT": ip.get("VGAT"), "ipsi_PV": ip.get("PV"),
                  "bilateral": d[d.kind == "bilateral"].norm.mean(), "contralateral": d[d.kind == "contralateral"].norm.mean()})
judge = pd.DataFrame(judge)
judge["holds"] = (judge.ipsi_VGAT < 0.1) & (judge.ipsi_PV < 0.1) & (judge.bilateral < 0.1) & (judge.contralateral > 0.8)
print("\n", judge.round(3).to_string(index=False))

**Result: partially reproduced.** Photoinhibition silences most activity, but not by more than 90%
in every case. With lick-early trials dropped, the normalised rate in experiment 2 is 0.077 to 0.097
for bilateral and 0.94 to 1.00 for contralateral photoinhibition, and 0.035 to 0.052 for
ipsilateral photoinhibition in PV x ReaChR mice, but 0.177 to 0.245 in VGAT-ChR2-EYFP mice. With
lick-early trials kept, all silencing values are higher (0.14 to 0.36). Experiment 1 gives 0.037 to
0.098 (dropped). Neuron counts are 172, 299 and 55 to 110 per condition, against 109 to 117, 294 to
301 and 52 to 102.

## ed2e-no-rebound

> No rebound excitation was detected on average after photoinhibition offset; a small fraction of
> neurons showed rebound, balanced by weak sustained inhibition in a larger fraction.

Criterion: experiment 2, photoinhibition 0.8 s at delay onset, lick right trials only, the six
400 ms windows of the legend; per neuron control against photoinhibition by two-tailed t-test,
p < 0.01. Reproduced when, in the first 400 ms and 400 to 800 ms after photoinhibition
(ipsilateral), the population mean rate is not higher than control (paired t-test across neurons,
p > 0.05, a reading) and fewer neurons have a significant increase than a significant decrease.
Neurons: putative pyramidal with at least 2 lick right trials in each condition (a standard choice);
lick-early dropped or kept (question 1).

In [ ]:
WIN = {"last 400 ms sample": (-2.1, -1.7), "first 400 ms light": (-1.7, -1.3), "last 400 ms light": (-1.3, -0.9),
       "first 400 ms after": (-0.9, -0.5), "400-800 ms after": (-0.5, -0.1), "first 400 ms response": (0.0, 0.4)}
e2 = U[(U.exp == "exp2") & (U.cls == "pyr")]
rows = []
for early in ("drop", "keep"):
    for c in ("left1", "right1", "bilateral8"):
        for u in e2.itertuples():
            tr = S[u.sid]["trials"]
            m = L.trial_mask(tr, early) & u.observed & (tr.instruction == "right").values
            k, p = m & (tr.cond == "control").values, m & (tr.cond == c).values
            if k.sum() < 2 or p.sum() < 2:
                continue
            for wn, (a, b) in WIN.items():
                cc = L.counts_in(u.spikes, tr.go.values[k], a, b) / (b - a)
                cp = L.counts_in(u.spikes, tr.go.values[p], a, b) / (b - a)
                pv = L.ttest(cp, cc)
                rows.append({"early_lick": early, "cond": c, "window": wn, "uid": u.uid, "ctrl": cc.mean(), "pi": cp.mean(),
                             "up": pv < 0.01 and cp.mean() > cc.mean(), "down": pv < 0.01 and cp.mean() < cc.mean()})
rb = pd.DataFrame(rows)
def summ(d):
    t = stats.ttest_rel(d.pi, d.ctrl)
    return pd.Series({"neurons": len(d), "ctrl_mean": d.ctrl.mean(), "pi_mean": d.pi.mean(), "paired_t": t.statistic, "paired_p": t.pvalue,
                      "n_up": int(d.up.sum()), "n_down": int(d.down.sum())})
rbs = rb.groupby(["early_lick", "cond", "window"], sort=False).apply(summ).reset_index()
print(rbs.round(4).to_string(index=False))
j = rbs[(rbs.cond == "left1") & rbs.window.isin(["first 400 ms after", "400-800 ms after"])].copy()
j["not_higher"] = ~((j.pi_mean > j.ctrl_mean) & (j.paired_p < 0.05))
j["up_lt_down"] = j.n_up < j.n_down
print("\nJudged windows (ipsilateral):")
print(j[["early_lick", "window", "pi_mean", "ctrl_mean", "paired_p", "n_up", "n_down", "not_higher", "up_lt_down"]].round(4).to_string(index=False))

**Result: not reproduced.** After ipsilateral photoinhibition the population mean rate is not higher
than control in either window (paired p 0.93 and 0.22 with lick-early trials dropped, 0.70 and 0.06
kept), but more neurons show a significant increase than a significant decrease: 27 against 7 in
the first 400 ms and 30 against 3 in 400 to 800 ms (dropped; 31 against 10 and 31 against 4 kept),
the reverse of the legend's balance. After bilateral photoinhibition the population mean rises (3.84
against 3.21 spikes/s, p = 0.002, in 400 to 800 ms).

## ed6a-bilateral-inconsistent-bias

> Bilateral photoinhibition biased movements inconsistently across mice and sessions, toward lick
> left in some and lick right in others.

Criterion: per session of the bilateral experiment, bias = (lick right minus lick left performance)
under bilateral photoinhibition minus the same in control; reproduced when, among sessions with a
bias significant by Fisher's exact test (p < 0.05), both signs occur, in at least two mice. Which
2 x 2 table enters Fisher's test is not given (question 7); readings (i) bilateral trials, trial
type by correct; (ii) licked side by condition (control against bilateral). Experiment 2 sessions
(bilateral photoinhibition of four spots per side); performance excludes lick-early trials
(convention P) and no-response trials (shown with them counted as errors as a variant).
Experiment 3 (one spot per side) is shown as a variant.

In [ ]:
def session_bias(tr, bil, ignore_as_error=False):
    outs = ("hit", "miss", "ignore") if ignore_as_error else ("hit", "miss")
    m = tr.behaving & ~tr.early_lick & tr.outcome.isin(outs)
    out = {}
    for c in ("control", bil):
        t = tr[m & (tr.cond == c)]
        r, l = t[t.instruction == "right"], t[t.instruction == "left"]
        out[c] = dict(pr=(r.outcome == "hit").mean(), pl=(l.outcome == "hit").mean(),
                      rh=int((r.outcome == "hit").sum()), rn=len(r), lh=int((l.outcome == "hit").sum()), ln=len(l),
                      lick_right=int(((t.instruction == "right") & (t.outcome == "hit")).sum() + ((t.instruction == "left") & (t.outcome != "hit")).sum()),
                      n=len(t))
    b, c = out[bil], out["control"]
    bias = (b["pr"] - b["pl"]) - (c["pr"] - c["pl"])
    p_i = stats.fisher_exact([[b["rh"], b["rn"] - b["rh"]], [b["lh"], b["ln"] - b["lh"]]]).pvalue
    p_ii = stats.fisher_exact([[b["lick_right"], b["n"] - b["lick_right"]], [c["lick_right"], c["n"] - c["lick_right"]]]).pvalue
    return bias, p_i, p_ii, b["n"]

rows = []
for s in S.values():
    for exp, bil in (("exp2", "bilateral8"), ("exp3", "bilateral2")):
        if s["exp"] != exp or not s["trials"].behaving.any():
            continue
        for ign in (False, True):
            bias, p_i, p_ii, n = session_bias(s["trials"], bil, ign)
            rows.append({"experiment": exp, "sid": s["sid"], "subject": s["subject"], "ignore_as_error": ign, "bilateral_trials": n,
                         "bias": bias, "p_i": p_i, "p_ii": p_ii})
b6 = pd.DataFrame(rows)
BIAS = b6[(b6.experiment == "exp2") & ~b6.ignore_as_error].set_index("sid")
print(b6[~b6.ignore_as_error].round(4).to_string(index=False))
for (exp, ign), d in b6.groupby(["experiment", "ignore_as_error"]):
    for col in ("p_i", "p_ii"):
        sig = d[d[col] < 0.05]
        pos, neg = sig[sig.bias > 0], sig[sig.bias < 0]
        print(f"{exp}, no-response as error {ign}, Fisher reading {col}: {len(sig)} significant sessions; "
              f"lick right bias {len(pos)} sessions in {pos.subject.nunique()} mice, lick left bias {len(neg)} sessions in {neg.subject.nunique()} mice; "
              f"both signs, in at least two mice in all: {len(pos) > 0 and len(neg) > 0 and sig.subject.nunique() >= 2}; "
              f"each sign in at least two mice: {pos.subject.nunique() >= 2 and neg.subject.nunique() >= 2}")

**Result: reproduced.** In experiment 2, 12 sessions (Fisher reading i) or 13 (reading ii) have a
significant bias; 7 or 8 of them are biased toward lick right, in 4 mice, and 5 toward lick left, in
4 mice, with no-response trials left out or counted as errors. Both signs therefore occur, each in
at least two mice. Experiment 3 (one spot per side) shows few significant sessions (2 or 3).

## fig2e-selectivity-recovery-times

> Selectivity recovered to 80% of control 538 ms after ipsilateral and 192 ms after contralateral
> photoinhibition, and did not recover after bilateral photoinhibition.

Criterion: convention D on experiment 2 (ipsilateral one spot, contralateral one spot, bilateral
four spots per side; light 0.8 s from delay onset, offset 0.9 s before the go cue). Bilateral, no
recovery means photoinhibition selectivity below 80% of control up to the go cue. Reproduced when
both times are within tolerance and bilateral does not recover, under one bin reading. The open
details are run as for fig1g.

In [ ]:
OFF2 = {"left1": -0.9, "right1": -0.9, "bilateral8": -0.9}
t0 = time.time()
rec2, courses2 = {}, {}
for early in ("drop", "keep"):
    rec2[early], courses2[early] = L.recovery_readings(e2, S, OFF2, SEL[early], early, no_recovery=("bilateral8",))
rec2 = pd.concat(rec2.values(), ignore_index=True).rename(columns={"early": "early_lick"})
rec2["ipsi_ok"] = (rec2.left1 - 538).abs() <= 178
rec2["contra_ok"] = (rec2.right1 - 192).abs() <= 114
rec2["bilateral_no_recovery"] = ~rec2.bilateral8_recovers_by_go
rec2["holds"] = rec2.ipsi_ok & rec2.contra_ok & rec2.bilateral_no_recovery
print(f"{len(rec2)} reading combinations in {time.time() - t0:.0f} s")
print(rec2[["early_lick", "pref", "trials_rule", "label", "bin", "S", "n", "left1", "left1_boot", "left1_sd", "right1", "right1_boot", "right1_sd",
            "bilateral8", "bilateral8_max_ratio_to_go", "holds"]].round(2).to_string(index=False))

In [ ]:
s2e = rec2.groupby(["early_lick", "pref", "trials_rule", "label"]).apply(
    lambda d: pd.Series({"combos": len(d), "holds": int(d.holds.sum()), "ipsi_ok": int(d.ipsi_ok.sum()), "contra_ok": int(d.contra_ok.sum()),
                         "bilateral_no_recovery": int(d.bilateral_no_recovery.sum()), "n_range": f"{d.n.min()}-{d.n.max()}"})).reset_index()
print(s2e.to_string(index=False))
print("\nIncluded neurons:", rec2.n.min(), "to", rec2.n.max(), "; paper 143 (tolerance 14.3)")

**Result: not reproduced.** Under no reading are both times within tolerance. Ipsilateral selectivity
reaches 80% of control 1000 to 2400 ms after offset, after the go cue, or not at all (56 of 256
readings), against the paper's 538 ms; contralateral selectivity does so 200 to 1300 ms after offset
(paper 192 ms). Bilateral selectivity stays below 80% of control up to the go cue in 235 of 256
readings (at most 0.22 to 0.35 of control). 221 to 293 neurons are included, against the paper's
143.

## fig2-recordings-recovery

> After ipsilateral (one spot) photoinhibition spike rate (more than 90%) and selectivity (more than
> 80%) recovered within 600 ms; contralateral photoinhibition hardly affected activity; after
> bilateral photoinhibition spike rates recovered on average but selectivity did not.

Criterion: experiment 2, at 600 ms after photoinhibition offset (0.3 s before the go cue). Spike
rate recovery (a) fewer than 10% of neurons with a significant change (convention R), (b)
population mean rate at least 90% of control. Selectivity (convention D) at least 80% of control.
Contralateral: fewer than 10% of neurons significant in every window during and after
photoinhibition. Bilateral: rate recovered as in (a) or (b), and selectivity below 80% of control
at the go cue. Reproduced when all hold under one combination of readings (R: W1, W2, X1, X2; D:
bins and the S readings). Open details (questions 1 to 4) as for fig1g.

In [ ]:
R2 = {}
for early in ("drop", "keep"):
    for X in ("X1", "X2"):
        for c in OFF2:
            R2[(early, X, c)] = L.rate_change_tests(e2, S, c, 0.4, early, min_rate=1 if X == "X1" else None, min_trials=3 if X == "X1" else None)
rate_rows = []
for (early, X, c), (P, info) in R2.items():
    inc = info.included.values & ~np.isnan(P).all(1)
    for lab in ("end", "centre"):
        tl = L.labels(0.4, lab)
        j = int(np.argmin(abs(tl - (-0.9 + 0.6))))
        during_after = (tl >= -1.7 + 0.2) & (tl <= 0) if lab == "centre" else (tl >= -1.7 + 0.4) & (tl <= 0)
        frac_t = np.nanmean(P[inc] < 0.01, axis=0)
        rate_rows.append({"early_lick": early, "exclusion": X, "cond": c, "label": lab, "n": int(inc.sum()),
                          "frac_sig_600ms": frac_t[j], "max_frac_sig_during_after": frac_t[during_after].max()})
rr = pd.DataFrame(rate_rows)
# population mean rate at 600 ms after offset, photoinhibition over control (same windows)
pop_rows = []
for early in ("drop", "keep"):
    for lab in ("end", "centre"):
        tl = L.labels(0.4, lab); j = int(np.argmin(abs(tl - (-0.3))))
        for c in OFF2:
            num = den = 0.0
            for u in e2.itertuples():
                tr = S[u.sid]["trials"]; m = L.trial_mask(tr, early) & u.observed
                k, p = m & (tr.cond == "control").values, m & (tr.cond == c).values
                if k.sum() < 2 or p.sum() < 2:
                    continue
                e = L.GRID_END[j]
                num += L.counts_in(u.spikes, tr.go.values[p], e - 0.4, e).mean(); den += L.counts_in(u.spikes, tr.go.values[k], e - 0.4, e).mean()
            pop_rows.append({"early_lick": early, "label": lab, "cond": c, "pop_rate_ratio_600ms": num / den})
pr = pd.DataFrame(pop_rows)
print(rr.round(3).to_string(index=False))
print(pr.round(3).to_string(index=False))

In [ ]:
# selectivity ratio (photoinhibition over control) at 600 ms after offset and at the go cue
sel_rows = []
for early in ("drop", "keep"):
    for key, (tl, cm) in courses2[early].items():
        w, pw, pt, sk, lab = key
        j600 = int(np.argmin(abs(tl - (-0.3)))); jgo = int(np.argmin(abs(tl - 0.0)))
        sel_rows.append({"early_lick": early, "bin": w, "pref": pw, "trials_rule": "per type" if pt else "per condition", "S": sk, "label": lab,
                         "ipsi_sel_ratio_600ms": cm["left1"][j600] / cm["control"][j600],
                         "bilateral_sel_ratio_go": cm["bilateral8"][jgo] / cm["control"][jgo]})
sr = pd.DataFrame(sel_rows)
out = []
for early in ("drop", "keep"):
    for lab in ("end", "centre"):
        for X in ("X1", "X2"):
            r = rr[(rr.early_lick == early) & (rr.label == lab) & (rr.exclusion == X)].set_index("cond")
            p = pr[(pr.early_lick == early) & (pr.label == lab)].set_index("cond")
            ipsi_rate = (r.loc["left1", "frac_sig_600ms"] < 0.10) or (p.loc["left1", "pop_rate_ratio_600ms"] >= 0.9)
            bil_rate = (r.loc["bilateral8", "frac_sig_600ms"] < 0.10) or (p.loc["bilateral8", "pop_rate_ratio_600ms"] >= 0.9)
            contra = r.loc["right1", "max_frac_sig_during_after"] < 0.10
            for (pw, pt), d in sr[(sr.early_lick == early) & (sr.label == lab)].groupby(["pref", "trials_rule"]):
                sel_ok = (d.ipsi_sel_ratio_600ms >= 0.8) & (d.bilateral_sel_ratio_go < 0.8)
                out.append({"early_lick": early, "label": lab, "pref": pw, "trials_rule": pt, "R_exclusion": X,
                            "ipsi_rate_recovered": ipsi_rate, "bilateral_rate_recovered": bil_rate, "contra_hardly_affected": contra,
                            "selectivity_parts_hold_in": f"{int(sel_ok.sum())}/{len(d)}",
                            "holds": bool(ipsi_rate and bil_rate and contra and sel_ok.any())})
o2 = pd.DataFrame(out)
print(o2.to_string(index=False))
print("\nIpsilateral selectivity ratio at 600 ms, range over readings:", sr.ipsi_sel_ratio_600ms.min().round(2), "to", sr.ipsi_sel_ratio_600ms.max().round(2))
print("Bilateral selectivity ratio at the go cue, range over readings:", sr.bilateral_sel_ratio_go.min().round(2), "to", sr.bilateral_sel_ratio_go.max().round(2))

**Result: partially reproduced.** Spike rates recover: 600 ms after offset 8% to 10% of neurons
differ from control after ipsilateral photoinhibition, and the population rate is 1.00 to 1.05 of
control (1.16 to 1.23 after bilateral photoinhibition). Selectivity after bilateral photoinhibition
stays low, 0.22 to 0.31 of control at the go cue. Two parts fail under every reading: ipsilateral
selectivity at 600 ms is 0.45 to 0.72 of control, not above 0.8; and contralateral photoinhibition
changes the rate of up to 15% to 19% of neurons in some window during or after the light, not fewer
than 10%.

## Activity space (convention A)

The claims on the coding direction use the sessions of the bilateral experiment with more than 5
neurons recorded simultaneously for more than 5 trials per condition (control, ipsilateral,
contralateral, bilateral). Here the neurons are putative pyramidal (convention U), observed in every
analysis trial, with more than 5 trials in each condition (the claims file's words; counting each
trial type is shown as a variant that does not decide an outcome), with lick-early trials dropped or
kept (question 1), and with the training set
a random half (H1) or third (H2) of the correct control trials (the claims file's readings). The
ipsilateral trials used to build modes 2 and 3 are split with the same fraction (question 8). One
random split per session (seeded by the session) is used; the spread over 10 seeds is shown as a
variant. The paper's 16 sessions (6 to 20 neurons, mean 11) could not be matched by any reading
(question 5).

In [ ]:
CONDS_A = ["control", "left1", "right1", "bilateral8"]
A = {}
t0 = time.time()
for per_type in (False, True):
    for early in ("drop", "keep"):
        sess = L.a_sessions(U, S, "exp2", CONDS_A, early, per_type)
        for frac, hname in ((0.5, "H1"), (1 / 3, "H2")):
            res = {}
            for sid, uids in sess.items():
                X, t = L.session_counts(U, S, sid, uids, early)
                rng = np.random.default_rng(int(sid.replace("_", "")) % 2 ** 32)
                res[sid] = L.activity_session(X, t, frac, CONDS_A, "left1", rng)
            A[("per type" if per_type else "per condition", early, hname)] = res
print(f"convention A on {len(A)} readings in {time.time() - t0:.0f} s")
cnt = pd.DataFrame([{"trials_rule": k[0], "early_lick": k[1], "H": k[2], "sessions": len(v), "mice": len({s.split('_')[0] for s in v}),
                     "neurons_min": min(r["n_neurons"] for r in v.values()), "neurons_max": max(r["n_neurons"] for r in v.values()),
                     "neurons_mean": np.mean([r["n_neurons"] for r in v.values()])} for k, v in A.items()])
print(cnt.round(1).to_string(index=False))
print("Paper: 16 sessions of 7 mice, 6-20 neurons (mean 11).")

## cd-stable-over-epochs

> The direction of the lick right minus lick left difference vector was stable over the sample and
> delay epochs (correlation 0.61 between late sample and late delay).

Criterion: late sample and late delay are the last 400 ms window of each epoch; Pearson correlation
of the two w vectors per session, mean over the sessions; reproduced when within the paper's s.e.m.
(0.56 to 0.66). w uses the correct control trials (all of them; the training set is shown as a
variant).

In [ ]:
rows = []
for k, v in A.items():
    for which in ("all", "train"):
        x = np.array([r[f"w_corr_{which}"] for r in v.values()])
        rows.append({"trials_rule": k[0], "early_lick": k[1], "H": k[2], "w_trials": which, "sessions": len(x),
                     "mean": x.mean(), "sem": x.std(ddof=1) / np.sqrt(len(x)), "within": abs(x.mean() - 0.61) <= 0.05})
cs = pd.DataFrame(rows)
print(cs.round(3).to_string(index=False))

**Result: reproduced.** With w from all correct control trials and sessions selected by trials per
condition, the mean correlation between late sample and late delay w is 0.563 (lick-early trials
dropped) and 0.588 (kept) over 24 sessions, both within 0.56 to 0.66. Built from the training trials
alone it is 0.534 to 0.554. Counting trials per trial type (a variant) gives 0.559 and 0.580.

## fig3c-cd-recovery-roc

> Along cd, trajectories recovered after ipsilateral photoinhibition (ROC 0.73 against 0.76 in
> control) and were little changed by contralateral photoinhibition (0.74), but were permanently
> altered by bilateral photoinhibition (0.58).

Criterion: convention A; ROC of cd'x between lick right and lick left trials in the last 400 ms of
the delay, per session, mean over sessions. Reproduced when all four are within the paper's
s.e.m. under H1 or H2; partially when control, ipsilateral and bilateral are and their order holds.

In [ ]:
EXP3C = {"control": (0.76, 0.03), "left1": (0.73, 0.02), "right1": (0.74, 0.03), "bilateral8": (0.58, 0.03)}
rows = []
for k, v in A.items():
    roc = pd.DataFrame([r["roc_go"] for r in v.values()])
    row = {"trials_rule": k[0], "early_lick": k[1], "H": k[2], "sessions": len(roc)}
    for c in CONDS_A:
        row[c] = roc[c].mean(); row[c + "_sem"] = roc[c].sem()
        row[c + "_ok"] = abs(roc[c].mean() - EXP3C[c][0]) <= EXP3C[c][1]
    row["all_four"] = all(row[c + "_ok"] for c in CONDS_A)
    row["partial"] = row["control_ok"] and row["left1_ok"] and row["bilateral8_ok"] and row["control"] >= row["left1"] > row["bilateral8"]
    rows.append(row)
r3c = pd.DataFrame(rows)
print(r3c.round(3).to_string(index=False))
# variant: spread over 10 random splits for one reading
k0 = ("per condition", "drop", "H1")
sp = []
for seed in range(10):
    vals = []
    for sid, uids in L.a_sessions(U, S, "exp2", CONDS_A, "drop", False).items():
        X, t = L.session_counts(U, S, sid, uids, "drop")
        r = L.activity_session(X, t, 0.5, CONDS_A, "left1", np.random.default_rng(1000 + seed + int(sid.replace("_", "")) % 2 ** 20))
        vals.append(r["roc_go"])
    sp.append(pd.DataFrame(vals).mean())
sp = pd.DataFrame(sp)
print("\nVariant, 10 random splits (per condition, lick-early dropped, H1): mean ROC per split, range")
print(sp.agg(["min", "max"]).round(3).to_string())

**Result: not reproduced.** The bilateral value is within tolerance under every reading (0.558 to
0.559 with trials per condition), but the control ROC is 0.649 to 0.677, below the paper's 0.76 by
more than its s.e.m., and the ipsilateral (0.631 to 0.652) and contralateral (0.672 to 0.704) values
are also below theirs. The order holds: ipsilateral and contralateral are close to control, and
bilateral is lowest. Over 10 random training splits the control mean ranges 0.641 to 0.686. The
analysis uses 24 sessions with 6 to 37 neurons, where the paper used 16 with 6 to 20.

## ed8-roc-during-photoinhibition

> During photoinhibition trajectory separation along cd fell to 0.54 (ipsilateral), 0.64
> (contralateral) and 0.54 (bilateral), against 0.72 in control.

Criterion: convention A with cd'x in a window during photoinhibition: (a) the last 400 ms before the
ramp, (b) the 400 ms window centred on the photostimulus, (c) the mean over the photostimulus. With
0.8 s of light from delay onset and a 0.2 s ramp, (a) and (b) are the same window (0.2 to 0.6 s
after delay onset). Reproduced when all four are within tolerance under a reading.

In [ ]:
EXP8 = {"control": (0.72, 0.02), "left1": (0.54, 0.03), "right1": (0.64, 0.03), "bilateral8": (0.54, 0.01)}
rows = []
for k, v in A.items():
    for wname, key in (("a/b", "roc_light_ab"), ("c", "roc_light_c")):
        roc = pd.DataFrame([r[key] for r in v.values()])
        row = {"trials_rule": k[0], "early_lick": k[1], "H": k[2], "window": wname}
        for c in CONDS_A:
            row[c] = roc[c].mean(); row[c + "_ok"] = abs(roc[c].mean() - EXP8[c][0]) <= EXP8[c][1]
        row["all_four"] = all(row[c + "_ok"] for c in CONDS_A)
        rows.append(row)
r8 = pd.DataFrame(rows)
print(r8.round(3).to_string(index=False))

**Result: partially reproduced.** During the light, separation along cd falls below control in every
condition: control 0.649 to 0.685, ipsilateral 0.49 to 0.58, contralateral 0.59 to 0.64, bilateral
0.48 to 0.55 (trials per condition). Under reading (c) with lick-early trials dropped and H1, the
ipsilateral (0.546), contralateral (0.626) and bilateral (0.535) values are within tolerance, but the
control value is below 0.70 under every reading, so no reading has all four.

## cd-captures-selectivity

> Projection on cd captured 65.6% of population selectivity and 36.4% of task-related activity
> variance over the sample and delay epochs; modes 1 to 5 together captured 98.5% of variance and
> 95.8% of selectivity.

Criterion: selectivity is the RMS of the trial-averaged lick right minus lick left rate over neurons
and time in the sample and delay epochs; variance the RMS of the baseline-subtracted
trial-averaged activity; captured fraction (a) RMS ratio or (b) its square; baseline (a) the
pre-sample period (the 400 ms window before sample onset), (b) the mean over the trial. Reproduced
when the cd values are within tolerance under a reading; the mode 1-5 values must hold for a full
reproduction, else partial. The trial-averaged activity is that of the correct control trials of
the training set, from which cd and the modes are built (question 8).

In [ ]:
rows = []
for k, v in A.items():
    for base in ("pre-sample", "trial mean"):
        if base == "pre-sample":
            cap = pd.DataFrame([r["captured"] for r in v.values() if r["captured"] is not None])
        else:
            cap = []
            for sid in v:
                uids = L.a_sessions(U, S, "exp2", CONDS_A, k[1], k[0] == "per type")[sid]
                X, t = L.session_counts(U, S, sid, uids, k[1])
                rng = np.random.default_rng(int(sid.replace("_", "")) % 2 ** 32)
                c_ = L.activity_session(X, t, 0.5 if k[2] == "H1" else 1 / 3, CONDS_A, "left1", rng, baseline="trial mean")["captured"]
                if c_ is not None:
                    cap.append(c_)
            cap = pd.DataFrame(cap)
        for ratio in ("RMS", "squared"):
            pw = 1 if ratio == "RMS" else 2
            m = (cap ** pw).mean() * 100
            rows.append({"trials_rule": k[0], "early_lick": k[1], "H": k[2], "baseline": base, "ratio": ratio,
                         "sel_cd": m.sel_cd, "var_cd": m.var_cd, "sel_15": m.sel_15, "var_15": m.var_15})
cap = pd.DataFrame(rows)
cap["cd_ok"] = ((cap.sel_cd - 65.6).abs() <= 5.1) & ((cap.var_cd - 36.4).abs() <= 6.3)
cap["modes_ok"] = ((cap.var_15 - 98.5).abs() <= 0.5) & ((cap.sel_15 - 95.8).abs() <= 1.2)
print(cap.round(1).to_string(index=False))

**Result: not reproduced.** cd captures more of the selectivity than the paper reports: 74.8% to 88.0%
(paper 65.6 +/- 5.1) over every reading. The variance captured is within tolerance only with
squared ratios and the pre-sample baseline (37.5 to 44.8%, paper 36.4 +/- 6.3), where the
selectivity is 74.8 to 78.2%. Modes 1 to 5 capture 90.6 to 96.9% of selectivity and 92.1 to 98.0%
of variance.

## fig3c-cd-predicts-movement and fig4-bilateral-cd-predicts-movement

> Activity along cd predicts the upcoming movement trial by trial; deviations toward the lick right
> trajectory predict more lick right responses, and vice versa. (fig3c)

> Even after bilateral photoinhibition destroys average selectivity, single-trial activity along cd
> predicts the upcoming movement. (fig4)

Criterion: for each session the decision boundary on the training control trials; for each held-out
control and perturbation trial the distance of cd'x from the boundary in the last 400 ms before the
go cue; trials pooled over sessions. Reproduced, for control and ipsilateral trials (fig3c) or for
bilateral trials (fig4), when the fraction of trials whose lick falls on the side of the boundary is
above 0.5 by a one-sided binomial test at p < 0.05, and lick right performance rises and lick left
performance falls with distance (Spearman correlation over bins of 4 of the right sign). Bins hold
at least one trial; bins with at least 5 trials are shown as a variant.

In [ ]:
def predicts(trials, cond, min_bin=1):
    d = trials[trials.cond == cond]
    pred = (d.d > 0) == d.licked_right
    k, n = int(pred.sum()), len(d)
    pb = stats.binomtest(k, n, 0.5, alternative="greater").pvalue
    d = d.assign(bin=np.floor(d.d / 4) * 4 + 2)
    out = {"n": n, "frac_pred": k / n, "binom_p": pb}
    for side, name in ((True, "right"), (False, "left")):
        g = d[d.right == side].groupby("bin").hit.agg(["mean", "size"])
        g = g[g["size"] >= min_bin]
        rho = stats.spearmanr(g.index, g["mean"]) if len(g) > 2 else None
        out[f"rho_{name}"] = rho.statistic if rho else np.nan; out[f"p_{name}"] = rho.pvalue if rho else np.nan
        out[f"bins_{name}"] = len(g)
    out["holds"] = (pb < 0.05) and out["rho_right"] > 0 and out["rho_left"] < 0
    return out
rows = []
for k, v in A.items():
    trials = pd.concat([r["trials"] for r in v.values()], ignore_index=True)
    for c in CONDS_A:
        for mb in (1, 5):
            rows.append({"trials_rule": k[0], "early_lick": k[1], "H": k[2], "cond": c, "min_trials_per_bin": mb, **predicts(trials, c, mb)})
pm = pd.DataFrame(rows)
print(pm.round(4).to_string(index=False))

**Result: fig3c-cd-predicts-movement reproduced; fig4-bilateral-cd-predicts-movement reproduced.**
The side of the decision boundary predicts the lick in 67% to 69% of held-out control trials (1890
to 2895 trials) and 63% to 68% of ipsilateral trials (261 to 448), and in 59% to 64% of bilateral
trials (777 to 1139); every binomial p is below 1e-6. Lick right performance rises with the
distance from the boundary (Spearman rho +0.28 to +0.99 for control and ipsilateral trials, +0.84
to +1.00 for bilateral ones), and lick left performance falls (rho -0.76 to -0.99), under every
reading, with bins of at least one or at least 5 trials.

## fig3d-persistent-mode and fig3e-ramping-mode

> A persistent mode, built to capture the sustained effect of ipsilateral perturbation, stays
> altered past movement onset but neither discriminates trial type nor predicts behaviour on control
> trials. (fig3d)

> A ramping mode capturing most remaining variance shows non-selective ramping in the delay epoch,
> does not predict behaviour, and is resistant to unilateral and bilateral perturbation. (fig3e)

Criteria: mode 2 is the difference of mean response vectors, ipsilateral perturbed minus control
lick right correct trials, at the go cue; mode 4 the first SVD eigenvector of the baseline-subtracted
PSTH matrix (sample and delay epochs) made orthogonal to modes 1 to 3. fig3d is reproduced when the
session mean ROC between lick right and lick left control trials is within 2 bootstrap s.e.m. of 0.5,
the fraction predicted is not above 0.5 by the one-sided binomial test, and the ipsilateral minus
control difference at the go cue is outside 2 s.e.m. of zero. fig3e is reproduced when the control
projection rises over the delay (positive in the session mean), its ROC and fraction predicted fail
the tests of fig3d, and the perturbed minus control difference at the go cue is within 2 bootstrap
s.e.m. of zero for ipsilateral, contralateral and bilateral trials. The sign of an SVD eigenvector is
arbitrary; mode 4 is oriented so that its weights sum to a positive number, and the size of the
change over the delay is shown either way (question 8). The bootstrap is over sessions (1000
resamples).

In [ ]:
def boot_sem(x, n=1000, seed=0):
    rng = np.random.default_rng(seed); x = np.asarray(x, float)
    return np.std([rng.choice(x, len(x)).mean() for _ in range(n)])
rows = []
for k, v0 in A.items():
    v = {sid: r for sid, r in v0.items() if r["m2"] is not None}
    for mode in ("m2", "m4"):
        roc = np.array([r[mode]["roc"] for r in v.values()])
        k_ok = sum(r[mode]["n_correct_pred"] for r in v.values()); n_all = sum(r[mode]["n_pred"] for r in v.values())
        row = {"trials_rule": k[0], "early_lick": k[1], "H": k[2], "mode": mode, "sessions": len(v), "dropped": len(v0) - len(v),
               "roc": roc.mean(), "roc_2sem": 2 * boot_sem(roc),
               "frac_pred": k_ok / n_all, "binom_p": stats.binomtest(k_ok, n_all, 0.5, alternative="greater").pvalue}
        row["roc_near_half"] = abs(row["roc"] - 0.5) <= row["roc_2sem"]
        for c in ("left1", "right1", "bilateral8"):
            dg = np.array([r[mode]["diff_go"][c] for r in v.values()])
            row[f"diff_{c}"] = dg.mean(); row[f"diff_{c}_2sem"] = 2 * boot_sem(dg)
        rise = np.array([r[mode]["rise_delay"] for r in v.values()])
        row["rise_delay"] = rise.mean(); row["rise_2sem"] = 2 * boot_sem(rise)
        rows.append(row)
md = pd.DataFrame(rows)
print(md.round(3).to_string(index=False))
m2 = md[md["mode"] == "m2"].copy()
m2["holds_3d"] = m2.roc_near_half & (m2.binom_p >= 0.05) & (m2.diff_left1.abs() > m2.diff_left1_2sem)
m4 = md[md["mode"] == "m4"].copy()
m4["holds_3e"] = (m4.rise_delay > 0) & m4.roc_near_half & (m4.binom_p >= 0.05) & \
    np.all([m4[f"diff_{c}"].abs() <= m4[f"diff_{c}_2sem"] for c in ("left1", "right1", "bilateral8")], axis=0)
print("\nfig3d:"); print(m2[["trials_rule", "early_lick", "H", "roc", "roc_2sem", "frac_pred", "binom_p", "diff_left1", "diff_left1_2sem", "holds_3d"]].round(3).to_string(index=False))
print("\nfig3e:"); print(m4[["trials_rule", "early_lick", "H", "rise_delay", "rise_2sem", "roc", "roc_2sem", "frac_pred", "binom_p",
                           "diff_left1", "diff_left1_2sem", "diff_right1", "diff_right1_2sem", "diff_bilateral8", "diff_bilateral8_2sem", "holds_3e"]].round(3).to_string(index=False))

**Result: fig3d-persistent-mode not reproduced; fig3e-ramping-mode not reproduced.** The persistent
mode is altered by ipsilateral photoinhibition up to the go cue (difference 1.08 to 1.75, more than
2 s.e.m. of 0.54 to 0.68), but it predicts the lick on control trials above chance (60% to 62% of
trials, binomial p below 1e-6), and under half the readings its ROC (0.41 to 0.43) is further than 2
s.e.m. from 0.5. The ramping mode is resistant to all three perturbations (differences within 2
s.e.m. under every reading) and its ROC is near 0.5 in most readings, but it does not rise over the
delay (change from start to end of the delay -0.73 to +0.11, with mode 4 oriented to a positive
weight sum; positive under one of the four readings), and it predicts the lick in 55% to 58% of control trials, above chance (p below 1e-6).

## ed6b-session-bias-predicted

> In each session the behavioural bias after bilateral photoinhibition followed the side of the
> control trajectory that the perturbed cd trajectory ended closer to.

Criterion: per session with bilateral photoinhibition, the mean cd'x of bilateral trials at the go
cue relative to the decision boundary (convention A), and the behavioural bias of
ed6a-bilateral-inconsistent-bias; reproduced when the signs agree in a majority of sessions, above
chance by a one-sided binomial test at p < 0.05.

In [ ]:
rows = []
for k, v in A.items():
    agree = []
    for sid, r in v.items():
        tb = r["trials"]; side = tb[tb.cond == "bilateral8"].d.mean()
        agree.append({"sid": sid, "side": side, "bias": BIAS.loc[sid, "bias"], "agree": np.sign(side) == np.sign(BIAS.loc[sid, "bias"])})
    ag = pd.DataFrame(agree)
    kk = int(ag.agree.sum())
    rows.append({"trials_rule": k[0], "early_lick": k[1], "H": k[2], "sessions": len(ag), "agree": kk,
                 "binom_p": stats.binomtest(kk, len(ag), 0.5, alternative="greater").pvalue})
b6b = pd.DataFrame(rows)
b6b["holds"] = (b6b.agree > b6b.sessions / 2) & (b6b.binom_p < 0.05)
print(b6b.round(4).to_string(index=False))
print("\nPer session, one reading (per condition, lick-early dropped, H1):")
print(pd.DataFrame([{"sid": s, "side": r["trials"][r["trials"].cond == "bilateral8"].d.mean(),
      "bias": BIAS.loc[s, "bias"]} for s, r in A[("per condition", "drop", "H1")].items()]).round(3).to_string(index=False))

**Result: not reproduced.** The side of the boundary at which the bilateral trials end agrees in sign
with the behavioural bias in 11 to 15 of 19 to 24 sessions, not above chance (one-sided binomial p
0.15 to 0.50) under any reading. The legend's five example sessions are not identified here.

## ed9c-cd-window-robust

> The recovery along cd after unilateral perturbation does not depend on the window used to compute
> cd (first or last 400 ms of the delay epoch).

Criterion: as fig3c-cd-recovery-roc with cd from the first or the last 400 ms of the delay epoch;
reproduced when, for both windows, the ipsilateral ROC at the end of the delay is within 0.05 of the
control ROC and the bilateral ROC is lower than the ipsilateral. A 400 ms window inside a 400 ms
period is a single window, so cd is w of that window.

In [ ]:
rows = []
for per_type in (False, True):
    for early in ("drop", "keep"):
        sess = L.a_sessions(U, S, "exp2", CONDS_A, early, per_type)
        for frac, hname in ((0.5, "H1"), (1 / 3, "H2")):
            for wname, win in (("first 400 ms", (-1.7, -1.3)), ("last 400 ms", (-0.4, 0.0))):
                roc = []
                for sid, uids in sess.items():
                    X, t = L.session_counts(U, S, sid, uids, early)
                    rng = np.random.default_rng(int(sid.replace("_", "")) % 2 ** 32)
                    roc.append(L.activity_session(X, t, frac, CONDS_A, "left1", rng, cd_window=win)["roc_go"])
                roc = pd.DataFrame(roc).mean()
                rows.append({"trials_rule": "per type" if per_type else "per condition", "early_lick": early, "H": hname, "cd_window": wname,
                             **roc.to_dict()})
r9c = pd.DataFrame(rows)
r9c["holds"] = ((r9c.left1 - r9c.control).abs() <= 0.05) & (r9c.bilateral8 < r9c.left1)
print(r9c.round(3).to_string(index=False))
print(r9c.groupby(["trials_rule", "early_lick", "H"]).holds.all().to_string())

**Result: reproduced.** With cd from the first or the last 400 ms of the delay, the ipsilateral ROC is
within 0.05 of the control ROC and the bilateral ROC is below the ipsilateral one under H1 and H2
with lick-early trials dropped, and under H2 with them kept (under H1 kept, the ipsilateral ROC with
the first window is 0.059 to 0.072 below control). The criterion allows either H reading, so the
claim holds under both lick-early readings.

## ed7e-roc-larger-photoinhibition

> Trajectory separation along cd at the end of the delay was 0.80 in control, 0.64 after
> ipsilateral and 0.68 after contralateral four-spot photoinhibition, and 0.54 after bilateral
> one-spot photoinhibition.

Criterion: convention A on experiment 3 sessions with more than 5 simultaneously recorded neurons
tested on more than 3 trials per condition (control, ipsilateral four spots, contralateral four
spots, bilateral one spot per side). Tolerances are the legend's s.e.m. (the bilateral 0.8 read as
0.08). Reproduced when all four are within tolerance under H1 or H2.

In [ ]:
CONDS_7 = ["control", "left4", "right4", "bilateral2"]
EXP7E = {"control": (0.80, 0.08), "left4": (0.64, 0.10), "right4": (0.68, 0.15), "bilateral2": (0.54, 0.08)}
rows, A7 = [], {}
for early in ("drop", "keep"):
    sess = L.a_sessions(U, S, "exp3", CONDS_7, early, False, min_trials=3)
    for frac, hname in ((0.5, "H1"), (1 / 3, "H2")):
        res = {}
        for sid, uids in sess.items():
            X, t = L.session_counts(U, S, sid, uids, early)
            res[sid] = L.activity_session(X, t, frac, CONDS_7, "left4", np.random.default_rng(int(sid.replace("_", "")) % 2 ** 32))
        A7[(early, hname)] = res
        roc = pd.DataFrame([r["roc_go"] for r in res.values()])
        row = {"early_lick": early, "H": hname, "sessions": len(roc), "mice": len({s.split('_')[0] for s in res}),
               "neurons": f"{min(r['n_neurons'] for r in res.values())}-{max(r['n_neurons'] for r in res.values())}"}
        for c in CONDS_7:
            row[c] = roc[c].mean(); row[c + "_sem"] = roc[c].sem(); row[c + "_ok"] = abs(roc[c].mean() - EXP7E[c][0]) <= EXP7E[c][1]
        row["all_four"] = all(row[c + "_ok"] for c in CONDS_7)
        rows.append(row)
r7e = pd.DataFrame(rows)
print(r7e.round(3).to_string(index=False))

**Result: partially reproduced.** Over 6 sessions of 3 mice (6 to 23 neurons), the ipsilateral
(0.60 to 0.67) and contralateral (0.69 to 0.71) four-spot values are within tolerance, and the
bilateral one-spot value (0.615 to 0.616) is within it with lick-early trials dropped (0.646 to
0.665 kept, outside). The control ROC (0.680 to 0.689) is below the paper's 0.80 by more than its
s.e.m., so no reading has all four; bilateral separation is not lower than ipsilateral here.

## ed7-larger-photoinhibition

> Bilateral photoinhibition with one spot per side changed selectivity more persistently than
> unilateral photoinhibition with four spots; selectivity recovered to 80% of control 490 ms after
> ipsilateral and 235 ms after contralateral four-spot photoinhibition, and did not recover after
> bilateral one-spot photoinhibition; recovery after unilateral photoinhibition was less robust with
> larger photoinhibition.

Criterion: convention D on experiment 3. Reproduced when both times are within tolerance,
bilateral one-spot selectivity stays below 80% of control at the end of the delay, and the
selectivity change at the go cue is larger (more negative) after bilateral one-spot than after
ipsilateral four-spot photoinhibition; "less robust with size" is judged by the ipsilateral
four-spot ROC at the end of the delay (ed7e) lower than the ipsilateral one-spot ROC (fig3c), by
sign. Open details as for fig1g.

In [ ]:
OFF3 = {"left4": -0.9, "right4": -0.9, "bilateral2": -0.9}
e3 = U[(U.exp == "exp3") & (U.cls == "pyr")]
rec3, courses3 = {}, {}
for early in ("drop", "keep"):
    rec3[early], courses3[early] = L.recovery_readings(e3, S, OFF3, SEL[early], early, no_recovery=("bilateral2",), per_types=(False,))
rec3 = pd.concat(rec3.values(), ignore_index=True).rename(columns={"early": "early_lick"})
dsel = []
for early in ("drop", "keep"):
    for key, (tl, cm) in courses3[early].items():
        jgo = int(np.argmin(abs(tl - 0.0)))
        dsel.append({"early_lick": early, "bin": key[0], "pref": key[1], "S": key[3], "label": key[4],
                     "dsel_go_bilateral2": cm["bilateral2"][jgo] - cm["control"][jgo], "dsel_go_left4": cm["left4"][jgo] - cm["control"][jgo]})
rec3 = rec3.merge(pd.DataFrame(dsel), on=["early_lick", "bin", "pref", "S", "label"])
rec3["ipsi_ok"] = (rec3.left4 - 490).abs() <= 280
rec3["contra_ok"] = (rec3.right4 - 235).abs() <= 156
rec3["n_ok"] = (rec3.n - 40).abs() <= 4
rec3["bil_no_recovery"] = ~rec3.bilateral2_recovers_by_go
rec3["bil_larger_change"] = rec3.dsel_go_bilateral2 < rec3.dsel_go_left4
rec3["holds_D"] = rec3.ipsi_ok & rec3.contra_ok & rec3.bil_no_recovery & rec3.bil_larger_change
print(rec3[["early_lick", "pref", "label", "bin", "S", "n", "left4", "left4_boot", "left4_sd", "right4", "right4_boot", "right4_sd",
            "bilateral2", "bilateral2_max_ratio_to_go", "dsel_go_bilateral2", "dsel_go_left4", "holds_D"]].round(2).to_string(index=False))
roc_cmp = []
for early in ("drop", "keep"):
    for h in ("H1", "H2"):
        r4 = r7e[(r7e.early_lick == early) & (r7e.H == h)].left4.iloc[0]
        r1 = r3c[(r3c.trials_rule == "per condition") & (r3c.early_lick == early) & (r3c.H == h)].left1.iloc[0]
        roc_cmp.append({"early_lick": early, "H": h, "roc_ipsi_4_spots": r4, "roc_ipsi_1_spot": r1, "less_robust": r4 < r1})
roc_cmp = pd.DataFrame(roc_cmp)
print("\n", roc_cmp.round(3).to_string(index=False))
s7 = rec3.groupby(["early_lick", "pref", "label"]).holds_D.agg(["sum", "size"]).reset_index()
s7 = s7.merge(roc_cmp.groupby("early_lick").less_robust.any().reset_index(), on="early_lick")
s7["holds"] = (s7["sum"] > 0) & s7.less_robust
print("\n", s7.to_string(index=False))
print("Included neurons:", rec3.n.min(), "to", rec3.n.max(), "; paper 40 (tolerance 4)")

**Result: blocked (methods_underspecified).** The outcome depends on the window label (question 4).
With centre labels every lick-early reading has readings that hold (all 16 with lick-early trials
dropped, 2 to 6 of 16 with them kept). With them dropped: ipsilateral four-spot
recovery 650 to 700 ms (paper 490 +/- 280), contralateral 250 to 350 ms (235 +/- 156), bilateral
one-spot selectivity at most 0.74 to 0.76 of control up to the go cue, and a selectivity change at
the go cue of -0.39 to -0.47 after bilateral against -0.05 to +0.14 after ipsilateral four-spot
photoinhibition. With end labels the ipsilateral time is 750 to 900 ms, outside tolerance. The
four-spot ipsilateral ROC is below the one-spot ROC under H1 (0.607 against 0.652 dropped), as
"less robust" requires. 58 to 62 neurons are included, against the paper's 40.

## ed1-monolithic-models-fail

> Monolithic network models (integrator, integrator with corrective feedback, trained random
> recurrent networks) do not recover the unperturbed trajectory after transient silencing of half
> their neurons.

Criterion: simulations with the parameters of the Methods and the Extended Data Fig. 1 legend
(simple integrator N = 100, tau 100 ms, one eigenvalue 0.99; FORCE network N = 400, tau 200 ms,
tanh, p = 0.1, gain 1.5, 30 iterations; silencing 0.2 to 1.0 s of 2 s); 1 ms step, fixed seeds, at
least 10 networks. Reproduced when, at 2 s, the output of each model differs from its unperturbed
output by more than 10% of the unperturbed selectivity, for the simple integrator and the
transient-input FORCE network. The FORCE target (a ramp to plus or minus 1 at 2 s), the feedback
weights (uniform from -1 to 1), the learning rate and the update every 2 ms are standard choices
the paper does not give. The integrator with corrective feedback and the tamed-chaos network lack
parameters in the text and are not built.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
t0 = time.time()
rows = []
for seed in range(10):
    sil = np.random.default_rng(100 + seed).choice(100, 50, replace=False)
    for name, fn in (("integrator, persistent input", lambda **k: L.simple_integrator(**k)),
                     ("integrator, transient input", lambda **k: L.simple_integrator(transient=True, **k)),
                     ("cascaded integrators, transient input", lambda **k: L.cascaded_integrator(**k))):
        _, o = fn(seed=seed); _, p = fn(seed=seed, silence=sil)
        sel = o[1][-1] - o[-1][-1]
        rows.append({"model": name, "seed": seed, "selectivity": sel,
                     "max_rel_diff": max(abs(p[1][-1] - o[1][-1]), abs(p[-1][-1] - o[-1][-1])) / abs(sel)})
with ProcessPoolExecutor(4) as ex:
    force = list(ex.map(L.force_test, range(10)))
for r in force:
    sel = r["unpert_right"] - r["unpert_left"]
    rows.append({"model": "FORCE network, transient input", "seed": r["seed"], "selectivity": sel,
                 "max_rel_diff": max(abs(r["pert_right"] - r["unpert_right"]), abs(r["pert_left"] - r["unpert_left"])) / abs(sel),
                 "train_err": r["train_err"]})
mono = pd.DataFrame(rows)
print(f"simulated in {time.time() - t0:.0f} s")
print(mono.round(3).to_string(index=False))
ms = mono.groupby("model").agg(networks=("seed", "size"), min_rel_diff=("max_rel_diff", "min"), median_rel_diff=("max_rel_diff", "median"),
                               fail_to_recover=("max_rel_diff", lambda x: int((x > 0.1).sum())))
print("\n", ms.round(3).to_string())

**Result: reproduced.** Each of 10 simple integrators ends 20% to 22% of its selectivity away from its
unperturbed output after silencing half of its units (persistent input), and each of 10 FORCE
networks trained to ramp after a transient input ends 13% to 76% away (median 41%); all exceed the
10% bound. The transient-input and cascaded integrators end about 50% away. The two models whose
parameters the text lacks were not built.

## fig6-modular-models-recover

> Modular networks of two coupled modules (attractor, integrator, trained recurrent network) recover
> after silencing of one module; the modular attractor drifts to a random fixed point when both
> modules are silenced.

Criterion: modular attractor and modular integrator with the Methods' parameters; 1 ms step with
Euler-Maruyama. Reproduced when, over 100 simulated trials per trial type, the modular attractor ends
at the instructed fixed point in more than 90% of trials after unilateral silencing and in 30% to
70% after bilateral silencing, and the modular integrator returns within 10% of its unperturbed
trajectory by 2 s after unilateral silencing. How the noise of variance 0.01 scales with the step is
not given (question 10): (em) white noise of intensity 0.01, increment sd sqrt(0.01 dt)/tau; (sample)
one sample of variance 0.01 per 1 ms step, integrated like an input. Both are run. A trial ends at
the instructed fixed point when both modules have the instructed unit above the other one at 2 s.

In [ ]:
rows = []
for noise in ("em", "sample"):
    for cond, sil in (("unperturbed", ()), ("unilateral", (0, 1)), ("bilateral", (0, 1, 2, 3))):
        ok = []
        for k, trial in enumerate(("right", "left")):
            _, R = L.modular_attractor(trial, silence=sil, seed=7 + k + 10 * len(sil), noise=noise, n_trials=100)
            end = R[:, -1]
            inst = (end[:, 0] > end[:, 1]) & (end[:, 2] > end[:, 3]) if trial == "right" else (end[:, 1] > end[:, 0]) & (end[:, 3] > end[:, 2])
            ok.append(inst)
        ok = np.concatenate(ok)
        rows.append({"noise": noise, "silencing": cond, "trials": len(ok), "instructed_fixed_point": ok.mean()})
att = pd.DataFrame(rows)
print(att.to_string(index=False))
mi = []
for trial in ("right", "left"):
    _, R0 = L.modular_integrator(trial); _, R1 = L.modular_integrator(trial, silence=(0, 1, 2, 3))
    sel0 = R0[-1, 0] - R0[-1, 1]; sel1 = R1[-1, 0] - R1[-1, 1]
    sel0b = R0[-1, 4] - R0[-1, 5]; sel1b = R1[-1, 4] - R1[-1, 5]
    mi.append({"trial": trial, "module1_unperturbed": sel0, "module1_silenced": sel1, "rel_diff_module1": abs(sel1 - sel0) / abs(sel0),
               "module2_unperturbed": sel0b, "module2_after": sel1b, "rel_diff_module2": abs(sel1b - sel0b) / abs(sel0b)})
mi = pd.DataFrame(mi)
print("\nModular integrator, right minus left integrator unit at 2 s, module 1 silenced 0.2-1.0 s:")
print(mi.round(4).to_string(index=False))
holds = {}
for noise in ("em", "sample"):
    a = att[att.noise == noise].set_index("silencing").instructed_fixed_point
    holds[noise] = bool(a["unilateral"] > 0.9 and 0.3 <= a["bilateral"] <= 0.7 and (mi.rel_diff_module1 <= 0.1).all())
print("\nHolds under each noise reading:", holds)

**Result: partially reproduced.** Under the per-step noise reading the modular attractor ends at the
instructed fixed point in 100% of trials after unilateral silencing and in 48% after bilateral
silencing, as the claim says; under the Euler-Maruyama reading the noise dominates and it ends there
in about half of the trials whatever the silencing (51% unperturbed, 50% unilateral, 43% bilateral).
The modular integrator with the printed matrix restores lick right trials (0.2% from unperturbed at
2 s) but not lick left ones (50% away): its recovery unit passes only right-preferring activity.
The integrator part holds for one trial type under both noise readings, so the claim holds in
part under both; the attractor part holds under one of them.

## Blocked claims

- fig1d-late-delay-ipsilateral-bias, fig1d-earlier-photoinhibition-minimal,
  ed3-behaviour-recovery-interval, fig2b-unilateral-performance, fig2b-bilateral-performance
  (data_missing). They rest on the behavioural photoinhibition sessions of 5 and 13 mice
  (photoinhibition on 25% of trials, light ending at the go cue, 1.3 s light with a variable delay,
  four-spot unilateral and two-spot bilateral protocols). The dandiset holds only the recording
  sessions (40% light trials), which lack those conditions (data check). The behaviour during the
  recordings is shown above for reference only.
- ed5-photoactivation-recovery, ed5-photoactivation-rate-changes (data_missing). No
  Tlx_PL56-cre x Ai32 mouse is in the dandiset (genotypes in the data check).
- ed2d-spatial-extent (data_missing). The recordings of the 4 untrained mice are not in the
  dandiset.
- ed9f-reaction-time (methods_underspecified). Reaction times need lick onsets; the files hold
  only `lick_trace`, an analogue signal in 5 s chunks, with no lick events. The rule for reading a
  lick from it is not in the claims file or the paper (question 9).
- fig5b-callosotomy-control-performance (data_missing). There are no sessions before the
  callosotomy, and no file marks the 7 mice as callosotomised.
- fig5b-callosotomy-unilateral-sensitivity, ed10-callosotomy-preparatory-activity,
  fig5-callosotomy-no-recovery, fig5f-callosotomy-cd-reduced (metadata_insufficient). The 33
  sessions of the 7 mice without a photostimulus record contain photoinhibition trials (silenced
  trials in the data check), but no file says which trials, where or when, so control and
  photoinhibition trials cannot be told apart. They hold 519 putative pyramidal units (paper 496).

## Summary

| claim | outcome |
|---|---|
| unit-classification | partially reproduced |
| selective-fraction | partially reproduced |
| fig1d-late-delay-ipsilateral-bias | blocked (data_missing) |
| fig1d-earlier-photoinhibition-minimal | blocked (data_missing) |
| ed3-behaviour-recovery-interval | blocked (data_missing) |
| ed4-return-to-trajectory | not reproduced |
| fig1f-spike-rate-recovery | reproduced |
| fig1g-selectivity-recovery | blocked (methods_underspecified) |
| ed5-photoactivation-recovery | blocked (data_missing) |
| ed5-photoactivation-rate-changes | blocked (data_missing) |
| ed2b-silencing-strength | partially reproduced |
| ed2d-spatial-extent | blocked (data_missing) |
| ed2e-no-rebound | not reproduced |
| fig2b-unilateral-performance | blocked (data_missing) |
| fig2b-bilateral-performance | blocked (data_missing) |
| ed6a-bilateral-inconsistent-bias | reproduced |
| fig2-recordings-recovery | partially reproduced |
| fig2e-selectivity-recovery-times | not reproduced |
| ed7-larger-photoinhibition | blocked (methods_underspecified) |
| ed7e-roc-larger-photoinhibition | partially reproduced |
| cd-stable-over-epochs | reproduced |
| cd-captures-selectivity | not reproduced |
| fig3c-cd-recovery-roc | not reproduced |
| ed8-roc-during-photoinhibition | partially reproduced |
| fig3c-cd-predicts-movement | reproduced |
| fig3d-persistent-mode | not reproduced |
| fig3e-ramping-mode | not reproduced |
| fig4-bilateral-cd-predicts-movement | reproduced |
| ed6b-session-bias-predicted | not reproduced |
| ed9f-reaction-time | blocked (methods_underspecified) |
| ed9c-cd-window-robust | reproduced |
| fig5b-callosotomy-control-performance | blocked (data_missing) |
| fig5b-callosotomy-unilateral-sensitivity | blocked (metadata_insufficient) |
| ed10-callosotomy-preparatory-activity | blocked (metadata_insufficient) |
| fig5-callosotomy-no-recovery | blocked (metadata_insufficient) |
| fig5f-callosotomy-cd-reduced | blocked (metadata_insufficient) |
| ed1-monolithic-models-fail | reproduced |
| fig6-modular-models-recover | partially reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's. Conditions are read from the laser traces and
events, as the trial table has none. Session 258250_20141202 (interleaved, overlapping trial
sequences) and the three experiment-1 sessions without a photostimulus record are left out of
trial-based analyses. Details the claims file leaves open are run under each reading, and several
counts of included neurons and sessions differ from the paper's (55 and 143 neurons for the
recovery times against 138 to 173 and 221 to 293 here; 16 sessions with 6 to 20 neurons for the
activity space against 24 with 6 to 37), so the paper's selections are not those of the claims
file. The simulations use standard choices for the integration step, the FORCE target and feedback,
and the seeds. Photoinhibition-trial random splits use one seed per session.

What was found about the dandiset. It holds the recordings of the paper's 12 control mice (58
sessions, 890 putative pyramidal units as in the paper) and 33 sessions of 7 further mice that
match the callosotomy experiment, but not the behavioural photoinhibition sessions, the
photoactivation and untrained mice, or any session before callosotomy. About 33 units of
intermediate spike width are missing (15 against 50). The photostimulus is not in the trial table;
it is recorded as laser traces per site and events, from which the site, the number of spots and
the timing can be read, but in 36 sessions (the 33 of the callosotomy mice and 3 of experiment 1)
it is missing although those sessions contain silenced trials. No file says which mice were
callosotomised. In lick-early trials the photostimulus event is logged at the restarted delay, up
to 1.6 s from the light. One session has interleaved overlapping trial sequences; 28 units have
spike times out of time order; the files' `cell_type` labels intermediate-width units as `FS`; the
behaviour-only file is empty.

In [ ]:
print(f"notebook ran in {time.time() - started:.0f} s")